In [ ]:
!pip install -q uv
!uv pip install --system moshi speechbrain sentencepiece pymcd resemblyzer

In [ ]:
import os
import torch
import torch.nn as nn
import torchaudio
import sentencepiece
import librosa
import numpy as np
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from torch.amp import autocast

try:
    from resemblyzer import VoiceEncoder, preprocess_wav
except ImportError:
    print("⚠️ Missing resemblyzer! Run: !pip install resemblyzer")

# ==========================================
# 0. THE DYNAMIC GRAPH KILLER
# ==========================================
import moshi.utils.compile

print("🔍 Hunting for Kyutai's CUDA Graph compiler...")
patched_any = False
for name, obj in vars(moshi.utils.compile).items():
    if isinstance(obj, type) and hasattr(obj, '__call__'):
        def make_bypass(orig_call):
            def bypass_call(self, *args, **kwargs):
                if hasattr(self, 'func'): return self.func(*args, **kwargs)
                return orig_call(self, *args, **kwargs)
            return bypass_call
        obj.__call__ = make_bypass(obj.__call__)
        print(f"✅ Successfully neutralized CUDA Graphs in: moshi.utils.compile.{name}")
        patched_any = True

# ==========================================
# 1. SETUP & EXACT CONFORMER ARCHITECTURE
# ==========================================
DEVICE_HOME = "cuda:0" 
DEVICE_WORK = "cuda:1" 

# 🚀 Points to the new Conformer weights you just trained
BRIDGE_PATH = "/kaggle/input/datasets/ahmedsadman099876/model-conformer-pre-ln/acoustic_bridge_best_v6_conformer.pt" 
TARGET_PT_PATH = "/kaggle/input/datasets/ahmedsadman099876/data899/0012_001164.pt"
USER_PROMPT_WAV = "/kaggle/input/datasets/ahmedsadman099876/recording/Recording.wav"
REFERENCE_WAV = "/kaggle/input/datasets/ahmedsadman099876/data458/0012_001164.wav"
OUTPUT_FILENAME = "/kaggle/working/final_bridged_eval_Conformer.wav"

class ConformerBlock(nn.Module):
    def __init__(self, d_model, n_heads=8, conv_kernel=31, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        
        self.norm2 = nn.LayerNorm(d_model)
        self.conv = nn.Sequential(
            nn.Conv1d(d_model, d_model * 2, kernel_size=1),
            nn.GLU(dim=1),
            nn.Conv1d(d_model, d_model, kernel_size=conv_kernel, padding=(conv_kernel-1)//2, groups=d_model),
            nn.BatchNorm1d(d_model),
            nn.SiLU(),
            nn.Conv1d(d_model, d_model, kernel_size=1),
            nn.Dropout(dropout)
        )
        
        self.norm3 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model)
        )

    def forward(self, x):
        res = x
        x = self.norm1(x)
        x, _ = self.attn(x, x, x)
        x = x + res
        
        res = x
        x = self.norm2(x).transpose(1, 2)
        x = self.conv(x).transpose(1, 2)
        x = x + res
        
        res = x
        x = self.norm3(x)
        x = self.ffn(x)
        return x + res

class StableConformerBridge(nn.Module):
    def __init__(self, vocab_size=2048, embed_dim=1024, id_dim=192, max_len=4096):
        super().__init__()
        self.cb0_embedding = nn.Embedding(vocab_size, embed_dim)
        self.id_projection = nn.Linear(id_dim, embed_dim)
        self.positional_encoding = nn.Embedding(max_len, embed_dim)
        
        self.blocks = nn.ModuleList([
            ConformerBlock(d_model=embed_dim) for _ in range(4)
        ])
        
        self.acoustic_heads = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, vocab_size)
            ) for _ in range(7)
        ])

    def apply_ppmq_adain(self, cb0_emb, id_vec, residual_factor, identity_factor):
        id_raw = self.id_projection(id_vec).unsqueeze(1)
        
        id_min, id_max = id_raw.min(dim=-1, keepdim=True)[0], id_raw.max(dim=-1, keepdim=True)[0]
        scale = (id_max - id_min) / 15
        id_quant = torch.round((id_raw - id_min) / (scale + 1e-8)) * scale + id_min
        
        
        cb0_mean = cb0_emb.mean(dim=-1, keepdim=True)
        cb0_std = torch.sqrt(cb0_emb.var(dim=-1, keepdim=True) + 1e-5)
        
        id_mean = id_quant.mean(dim=-1, keepdim=True)
        id_std = torch.sqrt(id_quant.var(dim=-1, keepdim=True) + 1e-5)
        
        identity_payload = cb0_std * ((id_quant - id_mean) / id_std) + cb0_mean
        return (cb0_emb * residual_factor) + (identity_payload * identity_factor)

    def forward(self, cb0_tokens, id_vec):
        if cb0_tokens.dim() == 3: cb0_tokens = cb0_tokens.view(cb0_tokens.shape[0], -1)
        seq_len = cb0_tokens.size(1)
        positions = torch.arange(0, seq_len, device=cb0_tokens.device).unsqueeze(0)
        
        x = self.cb0_embedding(cb0_tokens) + self.positional_encoding(positions)
        x = self.apply_ppmq_adain(x, id_vec, residual_factor=0.80, identity_factor=1.20)
        
        for block in self.blocks:
            x = block(x)
            
        logits_list = [head(x) for head in self.acoustic_heads]
        return torch.stack(logits_list, dim=2)

# ==========================================
# 2. LOAD MODELS & CONSTRUCT SANDWICH
# ==========================================
from moshi.models import loaders, LMGen

repo_id = "kyutai/moshiko-pytorch-bf16"
print("\nDownloading Mimi weights...")
mimi = loaders.get_mimi(hf_hub_download(repo_id, "tokenizer-e351c8d8-checkpoint125.safetensors"), device="cpu")
mimi = mimi.to(DEVICE_HOME)
mimi.set_num_codebooks(8)

print("Downloading Moshi weights (~15.4 GB)...")
moshi = loaders.get_moshi_lm(hf_hub_download(repo_id, "model.safetensors"), device="cpu")
text_tokenizer = sentencepiece.SentencePieceProcessor(hf_hub_download(repo_id, "tokenizer_spm_32k_3.model"))

print("\n🛠️ Sharding Model across GPUs...")
def shard_hook(module, args):
    try: target_dev = next(module.parameters()).device
    except StopIteration:
        try: target_dev = next(module.buffers()).device
        except StopIteration: return args
    if isinstance(args, tuple): return tuple(a.to(target_dev) if isinstance(a, torch.Tensor) else a for a in args)
    return args.to(target_dev) if isinstance(args, torch.Tensor) else args

mid = len(moshi.transformer.layers) // 2
for i in range(mid, len(moshi.transformer.layers)):
    moshi.transformer.layers[i].to(DEVICE_WORK)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)

moshi.emb.to(DEVICE_HOME)
moshi.text_emb.to(DEVICE_HOME)
for i in range(mid):
    moshi.transformer.layers[i].to(DEVICE_HOME)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)
if hasattr(moshi.transformer, "norm"):
    moshi.transformer.norm.to(DEVICE_HOME)
    moshi.transformer.norm.register_forward_pre_hook(shard_hook)
for name, module in moshi.named_children():
    if name not in ["emb", "text_emb", "transformer"]:
        module.to(DEVICE_HOME)
        module.register_forward_pre_hook(shard_hook)

for name, buf in moshi.named_buffers(recurse=True): buf.data = buf.data.to(DEVICE_HOME)
for attr in ['initial', 'delays', 'zero_token_id']:
    if hasattr(moshi, attr):
        t = getattr(moshi, attr)
        if isinstance(t, torch.Tensor): setattr(moshi, attr, t.to(DEVICE_HOME))

lm_gen = LMGen(moshi, temp=0.8, temp_text=0.7)

# ==========================================
# 3. PHASE 1: NATIVE GENERATION (SMART TRIMMER)
# ==========================================
try:
    wav, sr = torchaudio.load(USER_PROMPT_WAV)
    if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
    if sr != 24000: wav = torchaudio.functional.resample(wav, sr, 24000)
    wav = wav.unsqueeze(0).to(DEVICE_HOME)
    silence_padding = torch.zeros(1, 1, int(24000 * 0.5)).to(DEVICE_HOME)
    wav = torch.cat([wav, silence_padding], dim=-1)
    
except FileNotFoundError:
    wav = torch.zeros(1, 1, 24000 * 3).to(DEVICE_HOME)

print("\n⚡ Phase 1: Generating Base Tokens Natively...")
frame_size = mimi.frame_size
all_codes = []
with torch.no_grad(), mimi.streaming(batch_size=1):
    for offset in range(0, wav.shape[-1], frame_size):
        frame = wav[:, :, offset : offset + frame_size]
        if frame.shape[-1] < frame_size: frame = torch.nn.functional.pad(frame, (0, frame_size - frame.shape[-1]))
        all_codes.append(mimi.encode(frame))

collected_moshi_codes = []
generated_text_pieces = []

with torch.no_grad(), lm_gen.streaming(1), mimi.streaming(1):
    def process_step(tokens_out):
        text_token = tokens_out[0, 0].item()
        if text_token not in (0, 3): 
            generated_text_pieces.append(text_tokenizer.id_to_piece(text_token).replace(' ', ' '))
        collected_moshi_codes.append(tokens_out[:, 1:])

    for code in all_codes:
        tokens_out = lm_gen.step(code)
        if tokens_out is not None: process_step(tokens_out)

    print("Moshi is replying...")
    silence_code = torch.zeros_like(all_codes[0]).to(DEVICE_HOME)
    
    eos_detected = False
    tail_frames = 0
    MAX_TAIL = 4  
    
    for _ in range(120): 
        tokens_out = lm_gen.step(silence_code)
        if tokens_out is not None: 
            text_token = tokens_out[0, 0].item()
            if text_token in (0, 3):
                eos_detected = True
            process_step(tokens_out)
            
            if eos_detected:
                tail_frames += 1
                if tail_frames >= MAX_TAIL:
                    break
    
final_moshi_codes = torch.cat(collected_moshi_codes, dim=-1) 

print("\n" + "="*40)
print("MOSHI'S TEXT RESPONSE:")
print("".join(generated_text_pieces).strip() or "(No text generated)")
print("="*40 + "\n")

# ==========================================
# 4. PHASE 2: BRIDGING & DSP POST-PROCESSING
# ==========================================
print("🌉 Phase 2: Applying Conformer Bridge and DSP Filters...")

bridge = StableConformerBridge().to(DEVICE_HOME)
try:
    # Changed map_location just in case to avoid any DataParallel loading mismatches
    raw_state_dict = torch.load(BRIDGE_PATH, map_location="cpu", weights_only=True)
    # Strip module. prefix from DataParallel training
    new_state_dict = {k.replace('module.', ''): v for k, v in raw_state_dict.items()}
    bridge.load_state_dict(new_state_dict)
    bridge = bridge.to(DEVICE_HOME)
except Exception as e:
    print(f"⚠️ Could not load bridge weights: {e}")
bridge.eval()

# Load Target Identity
sample_data = torch.load(TARGET_PT_PATH, weights_only=True)
id_vec = sample_data["identity_vector"].view(1, -1).to(DEVICE_HOME, dtype=torch.float32)

with torch.no_grad():
    cb0_sequence = final_moshi_codes[:, 0, :] 
    
    with autocast('cuda', dtype=torch.bfloat16):
        new_logits = bridge(cb0_sequence, id_vec) 
    
    new_cb1_7 = torch.argmax(new_logits, dim=-1).transpose(1, 2) 
    
    hybrid_codes = final_moshi_codes.clone()
    hybrid_codes[:, 1:8, :] = new_cb1_7
    bridged_waveform = mimi.decode(hybrid_codes).to(torch.float32).cpu()

# DSP: Clean high-freq hiss and fade out tail
clean_waveform = torchaudio.functional.lowpass_biquad(bridged_waveform, sample_rate=24000, cutoff_freq=7500.0)

fade_samples = int(24000 * 0.2) 
if clean_waveform.shape[-1] > fade_samples:
    fade_curve = torch.linspace(1.0, 0.0, fade_samples) ** 2 
    clean_waveform[0, 0, -fade_samples:] *= fade_curve

torchaudio.save(OUTPUT_FILENAME, clean_waveform.squeeze(0), 24000)
print(f"🎉 Success! Conformer Bridged audio saved to: {OUTPUT_FILENAME}")
display(Audio(OUTPUT_FILENAME, rate=24000))

# ==========================================
# 5. PHASE 3: RESEARCH METRICS REPORT
# ==========================================
print("\n" + "="*50)
print("📊 FINAL RESEARCH METRICS REPORT")
print("="*50)

def get_prosody_stats(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    f0 = f0[~np.isnan(f0)]
    rms = librosa.feature.rms(y=y)[0]
    return {
        "mean_f0": np.mean(f0) if len(f0) > 0 else 0,
        "std_f0": np.std(f0) if len(f0) > 0 else 0,
        "mean_energy": np.mean(rms)
    }

target_stats = get_prosody_stats(REFERENCE_WAV)
bridge_stats = get_prosody_stats(OUTPUT_FILENAME)

print("1️⃣ PROSODY & EMPATHY ALIGNMENT")
print("-" * 45)
print(f"{'Metric':<18} | {'Target':<12} | {'Bridge Out':<12}")
print("-" * 45)
print(f"{'Mean Pitch (Hz)':<18} | {target_stats['mean_f0']:>12.2f} | {bridge_stats['mean_f0']:>12.2f}")
print(f"{'Pitch StdDev':<18} | {target_stats['std_f0']:>12.2f} | {bridge_stats['std_f0']:>12.2f}")
print(f"{'Mean Energy':<18} | {target_stats['mean_energy']:>12.6f} | {bridge_stats['mean_energy']:>12.6f}")
print("-" * 45)

try:
    from pymcd.mcd import Calculate_MCD
    mcd_toolbox = Calculate_MCD(MCD_mode="dtw")
    mcd_score = mcd_toolbox.calculate_mcd(REFERENCE_WAV, OUTPUT_FILENAME)
    print(f"\n2️⃣ ACOUSTIC DISTORTION")
    print("-" * 45)
    print(f"📉 Official DTW-MCD Score:      {mcd_score:.2f} dB")
except ImportError:
    print("📉 Please run `!pip install pymcd` to view MCD score.")

print(f"\n3️⃣ BIOMETRIC IDENTITY TRANSFER")
print("-" * 45)
try:
    # Initialize the VoiceEncoder
    encoder = VoiceEncoder()
    
    # Preprocess the audio files (Resemblyzer applies VAD and normalization)
    wav_target_processed = preprocess_wav(REFERENCE_WAV)
    wav_output_processed = preprocess_wav(OUTPUT_FILENAME)
    
    # Generate 256-dimensional d-vectors
    emb_target = encoder.embed_utterance(wav_target_processed)
    emb_generated = encoder.embed_utterance(wav_output_processed)
    
    # Calculate Cosine Similarity via inner product (vectors are L2 normalized)
    similarity = np.inner(emb_target, emb_generated)
    
    print(f"🧬 Identity Similarity Score: {similarity:.4f}")
    
    if similarity > 0.75:
        print("🟢 RESULT: Strong Identity Match!")
    elif similarity > 0.60:
        print("🟡 RESULT: Moderate Identity Match (Perceptually similar)")
    else:
        print("🔴 RESULT: Weak Match")
        
except Exception as e:
    print(f"⚠️ Identity Check Failed: {e}")

In [ ]:
import os
import torch
import torch.nn as nn
import torchaudio
import sentencepiece
import librosa
import numpy as np
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from torch.amp import autocast

try:
    from resemblyzer import VoiceEncoder, preprocess_wav
except ImportError:
    print("⚠️ Missing resemblyzer! Run: !pip install resemblyzer")

# ==========================================
# 0. THE DYNAMIC GRAPH KILLER
# ==========================================
import moshi.utils.compile

print("🔍 Hunting for Kyutai's CUDA Graph compiler...")
patched_any = False
for name, obj in vars(moshi.utils.compile).items():
    if isinstance(obj, type) and hasattr(obj, '__call__'):
        def make_bypass(orig_call):
            def bypass_call(self, *args, **kwargs):
                if hasattr(self, 'func'): return self.func(*args, **kwargs)
                return orig_call(self, *args, **kwargs)
            return bypass_call
        obj.__call__ = make_bypass(obj.__call__)
        print(f"✅ Successfully neutralized CUDA Graphs in: moshi.utils.compile.{name}")
        patched_any = True

# ==========================================
# 1. SETUP & EXACT TRUE SOUNDSTORM ARCHITECTURE
# ==========================================
DEVICE_HOME = "cuda:0" 
DEVICE_WORK = "cuda:1" 

# 🚀 Points to the new True SoundStorm weights
BRIDGE_PATH = "/kaggle/input/datasets/ahmedsadman099876/model9867/true_soundstorm_best.pt" 
TARGET_PT_PATH = "/kaggle/input/datasets/ahmedsadman099876/data899/0012_001164.pt"
USER_PROMPT_WAV = "/kaggle/input/datasets/ahmedsadman099876/recording/Recording.wav"
REFERENCE_WAV = "/kaggle/input/datasets/ahmedsadman099876/data458/0012_001164.wav"
OUTPUT_FILENAME = "/kaggle/working/final_bridged_eval_SoundStorm.wav"

class TrueSoundStormBridge(nn.Module):
    def __init__(self, vocab_size=2048, embed_dim=1024, id_dim=192, max_len=4096):
        super().__init__()
        self.cb0_embedding = nn.Embedding(vocab_size, embed_dim)
        self.id_projection = nn.Linear(id_dim, embed_dim)
        self.positional_encoding = nn.Embedding(max_len, embed_dim)
        
        self.cnn_prenet = nn.Sequential(
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            nn.GELU()
        )
        
        self.temporal_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=embed_dim, nhead=8, batch_first=True, dropout=0.1, norm_first=True 
            ), num_layers=2
        )
        
        self.rvq_feedback_embs = nn.ModuleList([nn.Embedding(vocab_size, embed_dim) for _ in range(6)])
        self.acoustic_heads = nn.ModuleList([nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, vocab_size)) for _ in range(7)])

    def apply_ppmq_adain(self, cb0_emb, id_vec, residual_factor, identity_factor):
        """Prosody-Preserving Mixed Quantization (PPMQ) - Tunable Mix"""
        id_raw = self.id_projection(id_vec).unsqueeze(1)
        id_min, id_max = id_raw.min(dim=-1, keepdim=True)[0], id_raw.max(dim=-1, keepdim=True)[0]
        scale = (id_max - id_min) / 15
        id_quant = torch.round((id_raw - id_min) / (scale + 1e-8)) * scale + id_min
        
        
        cb0_mean = cb0_emb.mean(dim=-1, keepdim=True)
        cb0_var = ((cb0_emb - cb0_mean) ** 2).mean(dim=-1, keepdim=True)
        cb0_std = torch.sqrt(cb0_var + 1e-5)
        
        id_mean = id_quant.mean(dim=-1, keepdim=True)
        id_var = ((id_quant - id_mean) ** 2).mean(dim=-1, keepdim=True)
        id_std = torch.sqrt(id_var + 1e-5)
        
        identity_payload = cb0_std * ((id_quant - id_mean) / id_std) + cb0_mean
        return (cb0_emb * residual_factor) + (identity_payload * identity_factor)

    def forward_infer(self, cb0_tokens, id_vec):
        """TRUE SOUNDSTORM INFERENCE: Fast Parallel Cascading."""
        if cb0_tokens.dim() == 3: cb0_tokens = cb0_tokens.view(cb0_tokens.shape[0], -1)
        seq_len = cb0_tokens.size(1)
        device = cb0_tokens.device
        
        positions = torch.arange(0, seq_len, device=device).unsqueeze(0)
        cb0_emb = self.cb0_embedding(cb0_tokens) + self.positional_encoding(positions)
        
        # 🛠️ THE DIALS: 80% Moshi Structure, 120% Target Identity Overdrive
        latent_stream = self.apply_ppmq_adain(cb0_emb, id_vec, residual_factor=0.80, identity_factor=1.20)
        
        logits_list = []
        for i in range(7):
            temp_stream = latent_stream.transpose(1, 2)
            temp_stream = self.cnn_prenet(temp_stream)
            temp_stream = temp_stream.transpose(1, 2)
            temporal_features = self.temporal_transformer(temp_stream)
            
            layer_logits = self.acoustic_heads[i](temporal_features)
            logits_list.append(layer_logits)
            
            if i < 6:
                predicted_tokens = layer_logits.argmax(dim=-1)
                latent_stream = latent_stream + self.rvq_feedback_embs[i](predicted_tokens)
                
        return torch.stack(logits_list, dim=2)

# ==========================================
# 2. LOAD MODELS & CONSTRUCT SANDWICH
# ==========================================
from moshi.models import loaders, LMGen

repo_id = "kyutai/moshiko-pytorch-bf16"
print("\nDownloading Mimi weights...")
mimi = loaders.get_mimi(hf_hub_download(repo_id, "tokenizer-e351c8d8-checkpoint125.safetensors"), device="cpu")
mimi = mimi.to(DEVICE_HOME)
mimi.set_num_codebooks(8)

print("Downloading Moshi weights (~15.4 GB)...")
moshi = loaders.get_moshi_lm(hf_hub_download(repo_id, "model.safetensors"), device="cpu")
text_tokenizer = sentencepiece.SentencePieceProcessor(hf_hub_download(repo_id, "tokenizer_spm_32k_3.model"))

print("\n🛠️ Sharding Model across GPUs...")
def shard_hook(module, args):
    try: target_dev = next(module.parameters()).device
    except StopIteration:
        try: target_dev = next(module.buffers()).device
        except StopIteration: return args
    if isinstance(args, tuple): return tuple(a.to(target_dev) if isinstance(a, torch.Tensor) else a for a in args)
    return args.to(target_dev) if isinstance(args, torch.Tensor) else args

mid = len(moshi.transformer.layers) // 2
for i in range(mid, len(moshi.transformer.layers)):
    moshi.transformer.layers[i].to(DEVICE_WORK)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)

moshi.emb.to(DEVICE_HOME)
moshi.text_emb.to(DEVICE_HOME)
for i in range(mid):
    moshi.transformer.layers[i].to(DEVICE_HOME)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)
if hasattr(moshi.transformer, "norm"):
    moshi.transformer.norm.to(DEVICE_HOME)
    moshi.transformer.norm.register_forward_pre_hook(shard_hook)
for name, module in moshi.named_children():
    if name not in ["emb", "text_emb", "transformer"]:
        module.to(DEVICE_HOME)
        module.register_forward_pre_hook(shard_hook)

for name, buf in moshi.named_buffers(recurse=True): buf.data = buf.data.to(DEVICE_HOME)
for attr in ['initial', 'delays', 'zero_token_id']:
    if hasattr(moshi, attr):
        t = getattr(moshi, attr)
        if isinstance(t, torch.Tensor): setattr(moshi, attr, t.to(DEVICE_HOME))

lm_gen = LMGen(moshi, temp=0.8, temp_text=0.7)

# ==========================================
# 3. PHASE 1: NATIVE GENERATION (SMART TRIMMER)
# ==========================================
try:
    wav, sr = torchaudio.load(USER_PROMPT_WAV)
    if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
    if sr != 24000: wav = torchaudio.functional.resample(wav, sr, 24000)
    wav = wav.unsqueeze(0).to(DEVICE_HOME)
    silence_padding = torch.zeros(1, 1, int(24000 * 0.5)).to(DEVICE_HOME)
    wav = torch.cat([wav, silence_padding], dim=-1)
except FileNotFoundError:
    wav = torch.zeros(1, 1, 24000 * 3).to(DEVICE_HOME)


print("\n⚡ Phase 1: Generating Base Tokens Natively...")
frame_size = mimi.frame_size
all_codes = []
with torch.no_grad(), mimi.streaming(batch_size=1):
    for offset in range(0, wav.shape[-1], frame_size):
        frame = wav[:, :, offset : offset + frame_size]
        if frame.shape[-1] < frame_size: frame = torch.nn.functional.pad(frame, (0, frame_size - frame.shape[-1]))
        all_codes.append(mimi.encode(frame))

collected_moshi_codes = []
generated_text_pieces = []

with torch.no_grad(), lm_gen.streaming(1), mimi.streaming(1):
    def process_step(tokens_out):
        text_token = tokens_out[0, 0].item()
        if text_token not in (0, 3): 
            generated_text_pieces.append(text_tokenizer.id_to_piece(text_token).replace(' ', ' '))
        collected_moshi_codes.append(tokens_out[:, 1:])

    for code in all_codes:
        tokens_out = lm_gen.step(code)
        if tokens_out is not None: process_step(tokens_out)

    print("Moshi is replying...")
    silence_code = torch.zeros_like(all_codes[0]).to(DEVICE_HOME)
    
    eos_detected = False
    tail_frames = 0
    MAX_TAIL = 4  
    
    for _ in range(120): 
        tokens_out = lm_gen.step(silence_code)
        if tokens_out is not None: 
            text_token = tokens_out[0, 0].item()
            if text_token in (0, 3):
                eos_detected = True
            process_step(tokens_out)
            
            if eos_detected:
                tail_frames += 1
                if tail_frames >= MAX_TAIL:
                    break
    
final_moshi_codes = torch.cat(collected_moshi_codes, dim=-1) 

print("\n" + "="*40)
print("MOSHI'S TEXT RESPONSE:")
print("".join(generated_text_pieces).strip() or "(No text generated)")
print("="*40 + "\n")

# ==========================================
# 4. PHASE 2: BRIDGING & DSP POST-PROCESSING
# ==========================================
print("🌉 Phase 2: Applying True SoundStorm Bridge and DSP Filters...")

bridge = TrueSoundStormBridge().to(DEVICE_HOME)
try:
    raw_state_dict = torch.load(BRIDGE_PATH, map_location=DEVICE_HOME, weights_only=True)
    new_state_dict = {k.replace('module.', ''): v for k, v in raw_state_dict.items()}
    bridge.load_state_dict(new_state_dict)
except Exception as e:
    print(f"⚠️ Could not load bridge weights: {e}")
bridge.eval()

# Load Target Identity
sample_data = torch.load(TARGET_PT_PATH, weights_only=True)
id_vec = sample_data["identity_vector"].view(1, -1).to(DEVICE_HOME, dtype=torch.float32)

with torch.no_grad():
    cb0_sequence = final_moshi_codes[:, 0, :] 
    
    # 🛠️ True SoundStorm Inference with bfloat16 safety
    with autocast('cuda', dtype=torch.bfloat16):
        new_logits = bridge.forward_infer(cb0_sequence, id_vec) 
    
    # Argmax and transpose to match Moshi's spatial bounds: (batch, 7, seq_len)
    new_cb1_7 = torch.argmax(new_logits, dim=-1).transpose(1, 2) 
    
    hybrid_codes = final_moshi_codes.clone()
    hybrid_codes[:, 1:8, :] = new_cb1_7
    bridged_waveform = mimi.decode(hybrid_codes).to(torch.float32).cpu()

# DSP: Clean high-freq hiss and fade out tail
clean_waveform = torchaudio.functional.lowpass_biquad(bridged_waveform, sample_rate=24000, cutoff_freq=7500.0)

fade_samples = int(24000 * 0.2) 
if clean_waveform.shape[-1] > fade_samples:
    fade_curve = torch.linspace(1.0, 0.0, fade_samples) ** 2 
    clean_waveform[0, 0, -fade_samples:] *= fade_curve

torchaudio.save(OUTPUT_FILENAME, clean_waveform.squeeze(0), 24000)
print(f"🎉 Success! SoundStorm Bridged audio saved to: {OUTPUT_FILENAME}")
display(Audio(OUTPUT_FILENAME, rate=24000))

# ==========================================
# 5. PHASE 3: RESEARCH METRICS REPORT
# ==========================================
print("\n" + "="*50)
print("📊 FINAL RESEARCH METRICS REPORT")
print("="*50)

def get_prosody_stats(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    f0 = f0[~np.isnan(f0)]
    rms = librosa.feature.rms(y=y)[0]
    return {
        "mean_f0": np.mean(f0) if len(f0) > 0 else 0,
        "std_f0": np.std(f0) if len(f0) > 0 else 0,
        "mean_energy": np.mean(rms)
    }

target_stats = get_prosody_stats(REFERENCE_WAV)
bridge_stats = get_prosody_stats(OUTPUT_FILENAME)

print("1️⃣ PROSODY & EMPATHY ALIGNMENT")
print("-" * 45)
print(f"{'Metric':<18} | {'Target':<12} | {'Bridge Out':<12}")
print("-" * 45)
print(f"{'Mean Pitch (Hz)':<18} | {target_stats['mean_f0']:>12.2f} | {bridge_stats['mean_f0']:>12.2f}")
print(f"{'Pitch StdDev':<18} | {target_stats['std_f0']:>12.2f} | {bridge_stats['std_f0']:>12.2f}")
print(f"{'Mean Energy':<18} | {target_stats['mean_energy']:>12.6f} | {bridge_stats['mean_energy']:>12.6f}")
print("-" * 45)

try:
    from pymcd.mcd import Calculate_MCD
    mcd_toolbox = Calculate_MCD(MCD_mode="dtw")
    mcd_score = mcd_toolbox.calculate_mcd(REFERENCE_WAV, OUTPUT_FILENAME)
    print(f"\n2️⃣ ACOUSTIC DISTORTION")
    print("-" * 45)
    print(f"📉 Official DTW-MCD Score:      {mcd_score:.2f} dB")
except ImportError:
    print("📉 Please run `!pip install pymcd` to view MCD score.")

print(f"\n3️⃣ BIOMETRIC IDENTITY TRANSFER")
print("-" * 45)
try:
    # Initialize the VoiceEncoder
    encoder = VoiceEncoder()
    
    # Preprocess the audio files (Resemblyzer applies VAD and normalization)
    wav_target_processed = preprocess_wav(REFERENCE_WAV)
    wav_output_processed = preprocess_wav(OUTPUT_FILENAME)
    
    # Generate 256-dimensional d-vectors
    emb_target = encoder.embed_utterance(wav_target_processed)
    emb_generated = encoder.embed_utterance(wav_output_processed)
    
    # Calculate Cosine Similarity via inner product (vectors are L2 normalized)
    similarity = np.inner(emb_target, emb_generated)
    
    print(f"🧬 Identity Similarity Score: {similarity:.4f}")
    
    if similarity > 0.75:
        print("🟢 RESULT: Strong Identity Match!")
    elif similarity > 0.60:
        print("🟡 RESULT: Moderate Identity Match (Perceptually similar)")
    else:
        print("🔴 RESULT: Weak Match")
        
except Exception as e:
    print(f"⚠️ Identity Check Failed: {e}")

In [ ]:
import os
import torch
import torch.nn as nn
import torchaudio
import sentencepiece
import librosa
import numpy as np
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from torch.amp import autocast

try:
    from resemblyzer import VoiceEncoder, preprocess_wav
except ImportError:
    print("⚠️ Missing resemblyzer! Run: !pip install resemblyzer")

# ==========================================
# 0. THE DYNAMIC GRAPH KILLER
# ==========================================
import moshi.utils.compile

print("🔍 Hunting for Kyutai's CUDA Graph compiler...")
patched_any = False
for name, obj in vars(moshi.utils.compile).items():
    if isinstance(obj, type) and hasattr(obj, '__call__'):
        def make_bypass(orig_call):
            def bypass_call(self, *args, **kwargs):
                if hasattr(self, 'func'): return self.func(*args, **kwargs)
                return orig_call(self, *args, **kwargs)
            return bypass_call
        obj.__call__ = make_bypass(obj.__call__)
        print(f"✅ Successfully neutralized CUDA Graphs in: moshi.utils.compile.{name}")
        patched_any = True

# ==========================================
# 1. SETUP & EXACT V2.3 TUNABLE ARCHITECTURE
# ==========================================
DEVICE_HOME = "cuda:0" 
DEVICE_WORK = "cuda:1" 

# 🚀 Points to the new V2.3 Tunable Mix weights
BRIDGE_PATH = "/kaggle/input/datasets/ahmedsadman099876/model3568/acoustic_bridge_best_v2-3_tunable.pt" 
TARGET_PT_PATH = "/kaggle/input/datasets/ahmedsadman099876/data899/0012_001164.pt"
USER_PROMPT_WAV = "/kaggle/input/datasets/ahmedsadman099876/recording/Recording.wav"
REFERENCE_WAV = "/kaggle/input/datasets/ahmedsadman099876/data458/0012_001164.wav"
OUTPUT_FILENAME = "/kaggle/working/final_bridged_eval_V2-3_Tunable.wav"

class AcousticDisentanglementBridge(nn.Module):
    def __init__(self, vocab_size=2048, embed_dim=1024, id_dim=192, max_len=4096):
        super().__init__()
        
        self.cb0_embedding = nn.Embedding(vocab_size, embed_dim)
        self.id_projection = nn.Linear(id_dim, embed_dim)
        self.positional_encoding = nn.Embedding(max_len, embed_dim)
        
        # Clean, continuous flow CNN
        self.cnn_prenet = nn.Sequential(
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            nn.GELU()
        )
        
        # Pre-LN Transformer 
        self.temporal_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=embed_dim, 
                nhead=8, 
                batch_first=True, 
                dropout=0.1,
                norm_first=True 
            ), 
            num_layers=2
        )
        
        self.acoustic_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, embed_dim // 2),
                nn.GELU(),
                nn.LayerNorm(embed_dim // 2),
                nn.Linear(embed_dim // 2, vocab_size)
            ) for _ in range(7)
        ])

    def apply_ppmq_adain(self, cb0_emb, id_vec, residual_factor, identity_factor):
        """Prosody-Preserving Mixed Quantization (PPMQ) - Tunable Mix"""
        id_raw = self.id_projection(id_vec).unsqueeze(1)
        id_min, id_max = id_raw.min(dim=-1, keepdim=True)[0], id_raw.max(dim=-1, keepdim=True)[0]
        scale = (id_max - id_min) / 15
        id_quant = torch.round((id_raw - id_min) / (scale + 1e-8)) * scale + id_min
        
        cb0_std, cb0_mean = cb0_emb.std(dim=-1, keepdim=True), cb0_emb.mean(dim=-1, keepdim=True)
        id_std, id_mean = id_quant.std(dim=-1, keepdim=True) + 1e-8, id_quant.mean(dim=-1, keepdim=True)
        
        # Shape target identity to safely fit Moshi's spatial boundaries
        identity_payload = cb0_std * ((id_quant - id_mean) / id_std) + cb0_mean
        
        # Apply the Mix
        return (cb0_emb * residual_factor) + (identity_payload * identity_factor)

    def forward(self, cb0_tokens, id_vec):
        if cb0_tokens.dim() == 3: cb0_tokens = cb0_tokens.view(cb0_tokens.shape[0], -1)
        seq_len = cb0_tokens.size(1)
        device = cb0_tokens.device
        
        positions = torch.arange(0, seq_len, device=device).unsqueeze(0)
        cb0_emb = self.cb0_embedding(cb0_tokens) + self.positional_encoding(positions)
        
        # 🛠️ THE DIALS: 80% Moshi Structure, 120% Target Identity Overdrive
        fused_emb = self.apply_ppmq_adain(cb0_emb, id_vec, residual_factor=0.80, identity_factor=1.20)
        
        fused_emb = fused_emb.transpose(1, 2)
        fused_emb = self.cnn_prenet(fused_emb)
        fused_emb = fused_emb.transpose(1, 2)
        
        temporal_features = self.temporal_transformer(fused_emb)
        
        logits_list = [head(temporal_features) for head in self.acoustic_heads]
        return torch.stack(logits_list, dim=2)

# ==========================================
# 2. LOAD MODELS & CONSTRUCT SANDWICH
# ==========================================
from moshi.models import loaders, LMGen

repo_id = "kyutai/moshiko-pytorch-bf16"
print("\nDownloading Mimi weights...")
mimi = loaders.get_mimi(hf_hub_download(repo_id, "tokenizer-e351c8d8-checkpoint125.safetensors"), device="cpu")
mimi = mimi.to(DEVICE_HOME)
mimi.set_num_codebooks(8)

print("Downloading Moshi weights (~15.4 GB)...")
moshi = loaders.get_moshi_lm(hf_hub_download(repo_id, "model.safetensors"), device="cpu")
text_tokenizer = sentencepiece.SentencePieceProcessor(hf_hub_download(repo_id, "tokenizer_spm_32k_3.model"))

print("\n🛠️ Sharding Model across GPUs...")
def shard_hook(module, args):
    try: target_dev = next(module.parameters()).device
    except StopIteration:
        try: target_dev = next(module.buffers()).device
        except StopIteration: return args
    if isinstance(args, tuple): return tuple(a.to(target_dev) if isinstance(a, torch.Tensor) else a for a in args)
    return args.to(target_dev) if isinstance(args, torch.Tensor) else args

mid = len(moshi.transformer.layers) // 2
for i in range(mid, len(moshi.transformer.layers)):
    moshi.transformer.layers[i].to(DEVICE_WORK)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)

moshi.emb.to(DEVICE_HOME)
moshi.text_emb.to(DEVICE_HOME)
for i in range(mid):
    moshi.transformer.layers[i].to(DEVICE_HOME)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)
if hasattr(moshi.transformer, "norm"):
    moshi.transformer.norm.to(DEVICE_HOME)
    moshi.transformer.norm.register_forward_pre_hook(shard_hook)
for name, module in moshi.named_children():
    if name not in ["emb", "text_emb", "transformer"]:
        module.to(DEVICE_HOME)
        module.register_forward_pre_hook(shard_hook)

for name, buf in moshi.named_buffers(recurse=True): buf.data = buf.data.to(DEVICE_HOME)
for attr in ['initial', 'delays', 'zero_token_id']:
    if hasattr(moshi, attr):
        t = getattr(moshi, attr)
        if isinstance(t, torch.Tensor): setattr(moshi, attr, t.to(DEVICE_HOME))

lm_gen = LMGen(moshi, temp=0.8, temp_text=0.7)

# ==========================================
# 3. PHASE 1: NATIVE GENERATION (SMART TRIMMER)
# ==========================================
try:
    wav, sr = torchaudio.load(USER_PROMPT_WAV)
    if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
    if sr != 24000: wav = torchaudio.functional.resample(wav, sr, 24000)
    wav = wav.unsqueeze(0).to(DEVICE_HOME)
    
    # 🛠️ THE FIX: Add 0.5 seconds of silence padding to prevent word clipping
    silence_padding = torch.zeros(1, 1, int(24000 * 0.5)).to(DEVICE_HOME)
    wav = torch.cat([wav, silence_padding], dim=-1)
except FileNotFoundError:
    wav = torch.zeros(1, 1, 24000 * 3).to(DEVICE_HOME)

print("\n⚡ Phase 1: Generating Base Tokens Natively...")
frame_size = mimi.frame_size
all_codes = []
with torch.no_grad(), mimi.streaming(batch_size=1):
    for offset in range(0, wav.shape[-1], frame_size):
        frame = wav[:, :, offset : offset + frame_size]
        if frame.shape[-1] < frame_size: frame = torch.nn.functional.pad(frame, (0, frame_size - frame.shape[-1]))
        all_codes.append(mimi.encode(frame))

collected_moshi_codes = []
generated_text_pieces = []

with torch.no_grad(), lm_gen.streaming(1), mimi.streaming(1):
    def process_step(tokens_out):
        text_token = tokens_out[0, 0].item()
        if text_token not in (0, 3): 
            generated_text_pieces.append(text_tokenizer.id_to_piece(text_token).replace(' ', ' '))
        collected_moshi_codes.append(tokens_out[:, 1:])

    for code in all_codes:
        tokens_out = lm_gen.step(code)
        if tokens_out is not None: process_step(tokens_out)

    print("Moshi is replying...")
    silence_code = torch.zeros_like(all_codes[0]).to(DEVICE_HOME)
    
    eos_detected = False
    tail_frames = 0
    MAX_TAIL = 4  
    
    for _ in range(120): 
        tokens_out = lm_gen.step(silence_code)
        if tokens_out is not None: 
            text_token = tokens_out[0, 0].item()
            if text_token in (0, 3):
                eos_detected = True
            process_step(tokens_out)
            
            if eos_detected:
                tail_frames += 1
                if tail_frames >= MAX_TAIL:
                    break
    
final_moshi_codes = torch.cat(collected_moshi_codes, dim=-1) 

print("\n" + "="*40)
print("MOSHI'S TEXT RESPONSE:")
print("".join(generated_text_pieces).strip() or "(No text generated)")
print("="*40 + "\n")

# ==========================================
# 4. PHASE 2: BRIDGING & DSP POST-PROCESSING
# ==========================================
print("🌉 Phase 2: Applying V2.3 Tunable Bridge and DSP Filters...")

bridge = AcousticDisentanglementBridge().to(DEVICE_HOME)
try:
    # Adding map_location="cpu" first is safer if loading from heavily distributed DP checkpoints
    raw_state_dict = torch.load(BRIDGE_PATH, map_location="cpu", weights_only=True)
    new_state_dict = {k.replace('module.', ''): v for k, v in raw_state_dict.items()}
    bridge.load_state_dict(new_state_dict)
    bridge = bridge.to(DEVICE_HOME)
except Exception as e:
    print(f"⚠️ Could not load bridge weights: {e}")
bridge.eval()

# Load Target Identity
sample_data = torch.load(TARGET_PT_PATH, weights_only=True)
id_vec = sample_data["identity_vector"].view(1, -1).to(DEVICE_HOME, dtype=torch.float32)

with torch.no_grad():
    cb0_sequence = final_moshi_codes[:, 0, :] 
    
    # 🛠️ THE FIX: Wrap the V2.3 inference in bfloat16 to match your training precision
    with autocast('cuda', dtype=torch.bfloat16):
        new_logits = bridge(cb0_sequence, id_vec) 
        
    new_cb1_7 = torch.argmax(new_logits, dim=-1).transpose(1, 2) 
    
    hybrid_codes = final_moshi_codes.clone()
    hybrid_codes[:, 1:8, :] = new_cb1_7
    bridged_waveform = mimi.decode(hybrid_codes).to(torch.float32).cpu()

# DSP: Clean high-freq hiss and fade out tail
clean_waveform = torchaudio.functional.lowpass_biquad(bridged_waveform, sample_rate=24000, cutoff_freq=7500.0)

fade_samples = int(24000 * 0.2) 
if clean_waveform.shape[-1] > fade_samples:
    fade_curve = torch.linspace(1.0, 0.0, fade_samples) ** 2 
    clean_waveform[0, 0, -fade_samples:] *= fade_curve

torchaudio.save(OUTPUT_FILENAME, clean_waveform.squeeze(0), 24000)
print(f"🎉 Success! V2.3 Tunable Bridged audio saved to: {OUTPUT_FILENAME}")
display(Audio(OUTPUT_FILENAME, rate=24000))

# ==========================================
# 5. PHASE 3: RESEARCH METRICS REPORT
# ==========================================
print("\n" + "="*50)
print("📊 FINAL RESEARCH METRICS REPORT")
print("="*50)

def get_prosody_stats(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    f0 = f0[~np.isnan(f0)]
    rms = librosa.feature.rms(y=y)[0]
    return {
        "mean_f0": np.mean(f0) if len(f0) > 0 else 0,
        "std_f0": np.std(f0) if len(f0) > 0 else 0,
        "mean_energy": np.mean(rms)
    }

target_stats = get_prosody_stats(REFERENCE_WAV)
bridge_stats = get_prosody_stats(OUTPUT_FILENAME)

print("1️⃣ PROSODY & EMPATHY ALIGNMENT")
print("-" * 45)
print(f"{'Metric':<18} | {'Target (Sad)':<12} | {'Bridge Out':<12}")
print("-" * 45)
print(f"{'Mean Pitch (Hz)':<18} | {target_stats['mean_f0']:>12.2f} | {bridge_stats['mean_f0']:>12.2f}")
print(f"{'Pitch StdDev':<18} | {target_stats['std_f0']:>12.2f} | {bridge_stats['std_f0']:>12.2f}")
print(f"{'Mean Energy':<18} | {target_stats['mean_energy']:>12.6f} | {bridge_stats['mean_energy']:>12.6f}")
print("-" * 45)

try:
    from pymcd.mcd import Calculate_MCD
    mcd_toolbox = Calculate_MCD(MCD_mode="dtw")
    mcd_score = mcd_toolbox.calculate_mcd(REFERENCE_WAV, OUTPUT_FILENAME)
    print(f"\n2️⃣ ACOUSTIC DISTORTION")
    print("-" * 45)
    print(f"📉 Official DTW-MCD Score:     {mcd_score:.2f} dB")
except ImportError:
    print("📉 Please run `!pip install pymcd` to view MCD score.")

print(f"\n3️⃣ BIOMETRIC IDENTITY TRANSFER")
print("-" * 45)
try:
    # Initialize the VoiceEncoder
    encoder = VoiceEncoder()
    
    # Preprocess the audio files (Resemblyzer applies VAD and normalization)
    wav_target_processed = preprocess_wav(REFERENCE_WAV)
    wav_output_processed = preprocess_wav(OUTPUT_FILENAME)
    
    # Generate 256-dimensional d-vectors
    emb_target = encoder.embed_utterance(wav_target_processed)
    emb_generated = encoder.embed_utterance(wav_output_processed)
    
    # Calculate Cosine Similarity via inner product (vectors are L2 normalized)
    similarity = np.inner(emb_target, emb_generated)
    
    print(f"🧬 Identity Similarity Score: {similarity:.4f}")
    
    if similarity > 0.75:
        print("🟢 RESULT: Strong Identity Match!")
    elif similarity > 0.60:
        print("🟡 RESULT: Moderate Identity Match (Perceptually similar)")
    else:
        print("🔴 RESULT: Weak Match")
        
except Exception as e:
    print(f"⚠️ Identity Check Failed: {e}")

In [ ]:
import os
import torch
import torch.nn as nn
import torchaudio
import sentencepiece
import librosa
import numpy as np
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from speechbrain.inference.speaker import SpeakerRecognition

# ==========================================
# 0. THE DYNAMIC GRAPH KILLER
# ==========================================
import moshi.utils.compile

print("🔍 Hunting for Kyutai's CUDA Graph compiler...")
patched_any = False
for name, obj in vars(moshi.utils.compile).items():
    if isinstance(obj, type) and hasattr(obj, '__call__'):
        def make_bypass(orig_call):
            def bypass_call(self, *args, **kwargs):
                if hasattr(self, 'func'): return self.func(*args, **kwargs)
                return orig_call(self, *args, **kwargs)
            return bypass_call
        obj.__call__ = make_bypass(obj.__call__)
        print(f"✅ Successfully neutralized CUDA Graphs in: moshi.utils.compile.{name}")
        patched_any = True

# ==========================================
# 1. SETUP, PATHS & SOUNDSTORM ARCHITECTURE
# ==========================================
DEVICE_HOME = "cuda:0" 
DEVICE_WORK = "cuda:1" 

# 🚀 Points to the best SoundStorm weights from the new training loop
BRIDGE_PATH = "/kaggle/input/datasets/ahmedsadman099876/model9878/acoustic_bridge_best_soundstorm.pt" 
TARGET_PT_PATH = "/kaggle/input/datasets/ahmedsadman099876/data899/0012_001164.pt"
USER_PROMPT_WAV = "/kaggle/input/datasets/ahmedsadman099876/recording/Recording.wav"
REFERENCE_WAV = "/kaggle/input/datasets/ahmedsadman099876/data458/0012_001164.wav"
OUTPUT_FILENAME = "/kaggle/working/final_bridged_eval_SoundStorm.wav"

class AcousticDisentanglementBridge(nn.Module):
    def __init__(self, vocab_size=2048, embed_dim=1024, id_dim=192, max_len=4096):
        super().__init__()
        self.cb0_embedding = nn.Embedding(vocab_size, embed_dim)
        self.id_projection = nn.Linear(id_dim, embed_dim)
        self.positional_encoding = nn.Embedding(max_len, embed_dim)
        
        self.base_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, batch_first=True, dropout=0.1), 
            num_layers=2
        )
        self.high_freq_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, batch_first=True, dropout=0.1), 
            num_layers=1
        )
        
        def create_head():
            return nn.Sequential(
                nn.Linear(embed_dim, embed_dim // 2),
                nn.GELU(),
                nn.LayerNorm(embed_dim // 2),
                nn.Linear(embed_dim // 2, vocab_size)
            )
            
        self.cb1_head = create_head()
        self.cb2_7_heads = nn.ModuleList([create_head() for _ in range(6)])

    def apply_ppmq_adain(self, cb0_emb, id_vec):
        id_raw = self.id_projection(id_vec).unsqueeze(1)
        id_min, id_max = id_raw.min(dim=-1, keepdim=True)[0], id_raw.max(dim=-1, keepdim=True)[0]
        scale = (id_max - id_min) / 15
        id_quant = torch.round((id_raw - id_min) / (scale + 1e-8)) * scale + id_min
        
        cb0_std, cb0_mean = cb0_emb.std(dim=-1, keepdim=True), cb0_emb.mean(dim=-1, keepdim=True)
        id_std, id_mean = id_quant.std(dim=-1, keepdim=True) + 1e-8, id_quant.mean(dim=-1, keepdim=True)
        
        return cb0_emb + ((cb0_std * ((id_quant - id_mean) / id_std) + cb0_mean) * 1)

    def forward(self, cb0_tokens, id_vec):
        if cb0_tokens.dim() == 3: cb0_tokens = cb0_tokens.view(cb0_tokens.shape[0], -1)
        seq_len = cb0_tokens.size(1)
        device = cb0_tokens.device
        
        positions = torch.arange(0, seq_len, device=device).unsqueeze(0)
        cb0_emb = self.cb0_embedding(cb0_tokens) + self.positional_encoding(positions)
        
        fused_emb = self.apply_ppmq_adain(cb0_emb, id_vec)
        fused_emb = torch.nan_to_num(fused_emb, nan=0.0, posinf=1.0, neginf=-1.0)
        
        base_features = self.base_transformer(fused_emb)
        cb1_logits = self.cb1_head(base_features)
        
        hf_features = self.high_freq_transformer(fused_emb + base_features)
        hf_logits = [head(hf_features) for head in self.cb2_7_heads]
        
        logits_list = [cb1_logits] + hf_logits
        return torch.stack(logits_list, dim=2)

# ==========================================
# 2. LOAD MODELS & CONSTRUCT SANDWICH
# ==========================================
from moshi.models import loaders, LMGen

repo_id = "kyutai/moshika-pytorch-bf16"
print("\nDownloading Mimi weights...")
mimi = loaders.get_mimi(hf_hub_download(repo_id, "tokenizer-e351c8d8-checkpoint125.safetensors"), device="cpu")
mimi = mimi.to(DEVICE_HOME)
mimi.set_num_codebooks(8)

print("Downloading Moshi weights (~15.4 GB)...")
moshi = loaders.get_moshi_lm(hf_hub_download(repo_id, "model.safetensors"), device="cpu")
text_tokenizer = sentencepiece.SentencePieceProcessor(hf_hub_download(repo_id, "tokenizer_spm_32k_3.model"))

print("\n🛠️ Sharding Model across GPUs...")
def shard_hook(module, args):
    try: target_dev = next(module.parameters()).device
    except StopIteration:
        try: target_dev = next(module.buffers()).device
        except StopIteration: return args
    if isinstance(args, tuple): return tuple(a.to(target_dev) if isinstance(a, torch.Tensor) else a for a in args)
    return args.to(target_dev) if isinstance(args, torch.Tensor) else args

mid = len(moshi.transformer.layers) // 2
for i in range(mid, len(moshi.transformer.layers)):
    moshi.transformer.layers[i].to(DEVICE_WORK)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)

moshi.emb.to(DEVICE_HOME)
moshi.text_emb.to(DEVICE_HOME)
for i in range(mid):
    moshi.transformer.layers[i].to(DEVICE_HOME)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)
if hasattr(moshi.transformer, "norm"):
    moshi.transformer.norm.to(DEVICE_HOME)
    moshi.transformer.norm.register_forward_pre_hook(shard_hook)
for name, module in moshi.named_children():
    if name not in ["emb", "text_emb", "transformer"]:
        module.to(DEVICE_HOME)
        module.register_forward_pre_hook(shard_hook)

for name, buf in moshi.named_buffers(recurse=True): buf.data = buf.data.to(DEVICE_HOME)
for attr in ['initial', 'delays', 'zero_token_id']:
    if hasattr(moshi, attr):
        t = getattr(moshi, attr)
        if isinstance(t, torch.Tensor): setattr(moshi, attr, t.to(DEVICE_HOME))

lm_gen = LMGen(moshi, temp=0.8, temp_text=0.7)

# ==========================================
# 3. PHASE 1: NATIVE GENERATION (SMART TRIMMER)
# ==========================================
try:
    wav, sr = torchaudio.load(USER_PROMPT_WAV)
    if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
    if sr != 24000: wav = torchaudio.functional.resample(wav, sr, 24000)
    wav = wav.unsqueeze(0).to(DEVICE_HOME)
except FileNotFoundError:
    wav = torch.zeros(1, 1, 24000 * 3).to(DEVICE_HOME)

print("\n⚡ Phase 1: Generating Base Tokens Natively...")
frame_size = mimi.frame_size
all_codes = []
with torch.no_grad(), mimi.streaming(batch_size=1):
    for offset in range(0, wav.shape[-1], frame_size):
        frame = wav[:, :, offset : offset + frame_size]
        if frame.shape[-1] < frame_size: frame = torch.nn.functional.pad(frame, (0, frame_size - frame.shape[-1]))
        all_codes.append(mimi.encode(frame))

collected_moshi_codes = []
generated_text_pieces = []

with torch.no_grad(), lm_gen.streaming(1), mimi.streaming(1):
    def process_step(tokens_out):
        text_token = tokens_out[0, 0].item()
        if text_token not in (0, 3): 
            generated_text_pieces.append(text_tokenizer.id_to_piece(text_token).replace(' ', ' '))
        collected_moshi_codes.append(tokens_out[:, 1:])

    for code in all_codes:
        tokens_out = lm_gen.step(code)
        if tokens_out is not None: process_step(tokens_out)

    print("Moshi is replying...")
    silence_code = torch.zeros_like(all_codes[0]).to(DEVICE_HOME)
    
    eos_detected = False
    tail_frames = 0
    MAX_TAIL = 4  
    
    for _ in range(120): 
        tokens_out = lm_gen.step(silence_code)
        if tokens_out is not None: 
            text_token = tokens_out[0, 0].item()
            if text_token in (0, 3):
                eos_detected = True
            process_step(tokens_out)
            
            if eos_detected:
                tail_frames += 1
                if tail_frames >= MAX_TAIL:
                    break
    
final_moshi_codes = torch.cat(collected_moshi_codes, dim=-1) 

print("\n" + "="*40)
print("MOSHI'S TEXT RESPONSE:")
print("".join(generated_text_pieces).strip() or "(No text generated)")
print("="*40 + "\n")

# ==========================================
# 4. PHASE 2: BRIDGING & DSP POST-PROCESSING
# ==========================================
print("🌉 Phase 2: Applying SoundStorm Bridge and DSP Filters...")

bridge = AcousticDisentanglementBridge().to(DEVICE_HOME)
raw_state_dict = torch.load(BRIDGE_PATH, map_location=DEVICE_HOME, weights_only=True)
bridge.load_state_dict(raw_state_dict)
bridge.eval()

sample_data = torch.load(TARGET_PT_PATH, weights_only=True)
id_vec = sample_data["identity_vector"].view(1, -1).to(DEVICE_HOME, dtype=torch.float32)

with torch.no_grad():
    cb0_sequence = final_moshi_codes[:, 0, :] 
    new_logits = bridge(cb0_sequence, id_vec) 
    new_cb1_7 = torch.argmax(new_logits, dim=-1).transpose(1, 2) 
    
    hybrid_codes = final_moshi_codes.clone()
    hybrid_codes[:, 1:8, :] = new_cb1_7
    bridged_waveform = mimi.decode(hybrid_codes).to(torch.float32).cpu()

clean_waveform = torchaudio.functional.lowpass_biquad(bridged_waveform, sample_rate=24000, cutoff_freq=7500.0)

fade_samples = int(24000 * 0.2) 
if clean_waveform.shape[-1] > fade_samples:
    fade_curve = torch.linspace(1.0, 0.0, fade_samples) ** 2 
    clean_waveform[0, 0, -fade_samples:] *= fade_curve

torchaudio.save(OUTPUT_FILENAME, clean_waveform.squeeze(0), 24000)
print(f"🎉 Success! SoundStorm Bridged audio saved to: {OUTPUT_FILENAME}")
display(Audio(OUTPUT_FILENAME, rate=24000))

# ==========================================
# 5. PHASE 3: RESEARCH METRICS REPORT
# ==========================================
print("\n" + "="*50)
print("📊 FINAL RESEARCH METRICS REPORT")
print("="*50)

def get_prosody_stats(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    f0 = f0[~np.isnan(f0)]
    rms = librosa.feature.rms(y=y)[0]
    return {
        "mean_f0": np.mean(f0) if len(f0) > 0 else 0,
        "std_f0": np.std(f0) if len(f0) > 0 else 0,
        "mean_energy": np.mean(rms)
    }

target_stats = get_prosody_stats(REFERENCE_WAV)
bridge_stats = get_prosody_stats(OUTPUT_FILENAME)

print("1️⃣ PROSODY & EMPATHY ALIGNMENT")
print("-" * 45)
print(f"{'Metric':<18} | {'Target (Sad)':<12} | {'Bridge Out':<12}")
print("-" * 45)
print(f"{'Mean Pitch (Hz)':<18} | {target_stats['mean_f0']:>12.2f} | {bridge_stats['mean_f0']:>12.2f}")
print(f"{'Pitch StdDev':<18} | {target_stats['std_f0']:>12.2f} | {bridge_stats['std_f0']:>12.2f}")
print(f"{'Mean Energy':<18} | {target_stats['mean_energy']:>12.6f} | {bridge_stats['mean_energy']:>12.6f}")
print("-" * 45)

try:
    from pymcd.mcd import Calculate_MCD
    mcd_toolbox = Calculate_MCD(MCD_mode="dtw")
    mcd_score = mcd_toolbox.calculate_mcd(REFERENCE_WAV, OUTPUT_FILENAME)
    print(f"\n2️⃣ ACOUSTIC DISTORTION")
    print("-" * 45)
    print(f"📉 Official DTW-MCD Score:     {mcd_score:.2f} dB")
except ImportError:
    print("📉 Please run `!pip install pymcd` to view MCD score.")

print(f"\n3️⃣ BIOMETRIC IDENTITY TRANSFER")
print("-" * 45)
try:
    verification = SpeakerRecognition.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb", 
        run_opts={"device": DEVICE_HOME}
    )
    wav_target = verification.load_audio(REFERENCE_WAV)
    wav_output = verification.load_audio(OUTPUT_FILENAME)
    
    emb1 = verification.encode_batch(wav_target)
    emb2 = verification.encode_batch(wav_output)
    
    similarity = torch.nn.functional.cosine_similarity(emb1.flatten(), emb2.flatten(), dim=0).item()
    print(f"🧬 ECAPA-TDNN Similarity:      {similarity:.4f}")
except Exception as e:
    print(f"⚠️ Identity Check Failed: {e}")

In [ ]:
import os
import torch
import torch.nn as nn
import torchaudio
import sentencepiece
import librosa
import numpy as np
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from speechbrain.inference.speaker import SpeakerRecognition

# ==========================================
# 0. THE DYNAMIC GRAPH KILLER
# ==========================================
import moshi.utils.compile

print("🔍 Hunting for Kyutai's CUDA Graph compiler...")
patched_any = False
for name, obj in vars(moshi.utils.compile).items():
    if isinstance(obj, type) and hasattr(obj, '__call__'):
        def make_bypass(orig_call):
            def bypass_call(self, *args, **kwargs):
                if hasattr(self, 'func'): return self.func(*args, **kwargs)
                return orig_call(self, *args, **kwargs)
            return bypass_call
        obj.__call__ = make_bypass(obj.__call__)
        print(f"✅ Successfully neutralized CUDA Graphs in: moshi.utils.compile.{name}")
        patched_any = True

# ==========================================
# 1. SETUP, PATHS & DILATED TCN ARCHITECTURE
# ==========================================
DEVICE_HOME = "cuda:0" 
DEVICE_WORK = "cuda:1" 

# 🚀 Points to the best TCN weights from the new training loop
BRIDGE_PATH = "/kaggle/input/datasets/ahmedsadman099876/model7890/acoustic_bridge_best_tcn.pt" 
TARGET_PT_PATH = "/kaggle/input/datasets/ahmedsadman099876/data899/0012_001164.pt"
USER_PROMPT_WAV = "/kaggle/input/datasets/ahmedsadman099876/recording/Recording.wav"
REFERENCE_WAV = "/kaggle/input/datasets/ahmedsadman099876/data458/0012_001164.wav"
OUTPUT_FILENAME = "/kaggle/working/final_bridged_eval_TCN.wav"

class ResidualDilatedConv1d(nn.Module):
    def __init__(self, channels, kernel_size, dilation):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.norm1 = nn.BatchNorm1d(channels)
        self.gelu = nn.GELU()
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.norm2 = nn.BatchNorm1d(channels)

    def forward(self, x):
        residual = x
        x = self.gelu(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        return self.gelu(x + residual)

class AcousticDisentanglementBridge(nn.Module):
    def __init__(self, vocab_size=2048, embed_dim=1024, id_dim=192, max_len=4096):
        super().__init__()
        self.cb0_embedding = nn.Embedding(vocab_size, embed_dim)
        self.id_projection = nn.Linear(id_dim, embed_dim)
        self.positional_encoding = nn.Embedding(max_len, embed_dim)
        
        self.tcn_backbone = nn.Sequential(
            ResidualDilatedConv1d(embed_dim, kernel_size=3, dilation=1),
            ResidualDilatedConv1d(embed_dim, kernel_size=3, dilation=2),
            ResidualDilatedConv1d(embed_dim, kernel_size=3, dilation=4),
            ResidualDilatedConv1d(embed_dim, kernel_size=3, dilation=8),
            ResidualDilatedConv1d(embed_dim, kernel_size=3, dilation=16)
        )
        
        self.acoustic_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, embed_dim // 2),
                nn.GELU(),
                nn.LayerNorm(embed_dim // 2),
                nn.Linear(embed_dim // 2, vocab_size)
            ) for _ in range(7)
        ])

    def apply_ppmq_adain(self, cb0_emb, id_vec):
        id_raw = self.id_projection(id_vec).unsqueeze(1)
        id_min, id_max = id_raw.min(dim=-1, keepdim=True)[0], id_raw.max(dim=-1, keepdim=True)[0]
        scale = (id_max - id_min) / 15
        id_quant = torch.round((id_raw - id_min) / (scale + 1e-8)) * scale + id_min
        
        cb0_std, cb0_mean = cb0_emb.std(dim=-1, keepdim=True), cb0_emb.mean(dim=-1, keepdim=True)
        id_std, id_mean = id_quant.std(dim=-1, keepdim=True) + 1e-8, id_quant.mean(dim=-1, keepdim=True)
        
        return cb0_emb + ((cb0_std * ((id_quant - id_mean) / id_std) + cb0_mean) * 0.7)

    def forward(self, cb0_tokens, id_vec):
        if cb0_tokens.dim() == 3: cb0_tokens = cb0_tokens.view(cb0_tokens.shape[0], -1)
        seq_len = cb0_tokens.size(1)
        device = cb0_tokens.device
        
        positions = torch.arange(0, seq_len, device=device).unsqueeze(0)
        cb0_emb = self.cb0_embedding(cb0_tokens) + self.positional_encoding(positions)
        
        fused_emb = self.apply_ppmq_adain(cb0_emb, id_vec)
        fused_emb = torch.nan_to_num(fused_emb, nan=0.0, posinf=1.0, neginf=-1.0)
        
        fused_emb = fused_emb.transpose(1, 2)
        temporal_features = self.tcn_backbone(fused_emb)
        temporal_features = temporal_features.transpose(1, 2)
        
        logits_list = [head(temporal_features) for head in self.acoustic_heads]
        return torch.stack(logits_list, dim=2)

# ==========================================
# 2. LOAD MODELS & CONSTRUCT SANDWICH
# ==========================================
from moshi.models import loaders, LMGen

repo_id = "kyutai/moshika-pytorch-bf16"
print("\nDownloading Mimi weights...")
mimi = loaders.get_mimi(hf_hub_download(repo_id, "tokenizer-e351c8d8-checkpoint125.safetensors"), device="cpu")
mimi = mimi.to(DEVICE_HOME)
mimi.set_num_codebooks(8)

print("Downloading Moshi weights (~15.4 GB)...")
moshi = loaders.get_moshi_lm(hf_hub_download(repo_id, "model.safetensors"), device="cpu")
text_tokenizer = sentencepiece.SentencePieceProcessor(hf_hub_download(repo_id, "tokenizer_spm_32k_3.model"))

print("\n🛠️ Sharding Model across GPUs...")
def shard_hook(module, args):
    try: target_dev = next(module.parameters()).device
    except StopIteration:
        try: target_dev = next(module.buffers()).device
        except StopIteration: return args
    if isinstance(args, tuple): return tuple(a.to(target_dev) if isinstance(a, torch.Tensor) else a for a in args)
    return args.to(target_dev) if isinstance(args, torch.Tensor) else args

mid = len(moshi.transformer.layers) // 2
for i in range(mid, len(moshi.transformer.layers)):
    moshi.transformer.layers[i].to(DEVICE_WORK)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)

moshi.emb.to(DEVICE_HOME)
moshi.text_emb.to(DEVICE_HOME)
for i in range(mid):
    moshi.transformer.layers[i].to(DEVICE_HOME)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)
if hasattr(moshi.transformer, "norm"):
    moshi.transformer.norm.to(DEVICE_HOME)
    moshi.transformer.norm.register_forward_pre_hook(shard_hook)
for name, module in moshi.named_children():
    if name not in ["emb", "text_emb", "transformer"]:
        module.to(DEVICE_HOME)
        module.register_forward_pre_hook(shard_hook)

for name, buf in moshi.named_buffers(recurse=True): buf.data = buf.data.to(DEVICE_HOME)
for attr in ['initial', 'delays', 'zero_token_id']:
    if hasattr(moshi, attr):
        t = getattr(moshi, attr)
        if isinstance(t, torch.Tensor): setattr(moshi, attr, t.to(DEVICE_HOME))

lm_gen = LMGen(moshi, temp=0.8, temp_text=0.7)

# ==========================================
# 3. PHASE 1: NATIVE GENERATION (SMART TRIMMER)
# ==========================================
try:
    wav, sr = torchaudio.load(USER_PROMPT_WAV)
    if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
    if sr != 24000: wav = torchaudio.functional.resample(wav, sr, 24000)
    wav = wav.unsqueeze(0).to(DEVICE_HOME)
except FileNotFoundError:
    wav = torch.zeros(1, 1, 24000 * 3).to(DEVICE_HOME)

print("\n⚡ Phase 1: Generating Base Tokens Natively...")
frame_size = mimi.frame_size
all_codes = []
with torch.no_grad(), mimi.streaming(batch_size=1):
    for offset in range(0, wav.shape[-1], frame_size):
        frame = wav[:, :, offset : offset + frame_size]
        if frame.shape[-1] < frame_size: frame = torch.nn.functional.pad(frame, (0, frame_size - frame.shape[-1]))
        all_codes.append(mimi.encode(frame))

collected_moshi_codes = []
generated_text_pieces = []

with torch.no_grad(), lm_gen.streaming(1), mimi.streaming(1):
    def process_step(tokens_out):
        text_token = tokens_out[0, 0].item()
        if text_token not in (0, 3): 
            generated_text_pieces.append(text_tokenizer.id_to_piece(text_token).replace(' ', ' '))
        collected_moshi_codes.append(tokens_out[:, 1:])

    for code in all_codes:
        tokens_out = lm_gen.step(code)
        if tokens_out is not None: process_step(tokens_out)

    print("Moshi is replying...")
    silence_code = torch.zeros_like(all_codes[0]).to(DEVICE_HOME)
    
    eos_detected = False
    tail_frames = 0
    MAX_TAIL = 4  
    
    for _ in range(120): 
        tokens_out = lm_gen.step(silence_code)
        if tokens_out is not None: 
            text_token = tokens_out[0, 0].item()
            if text_token in (0, 3):
                eos_detected = True
            process_step(tokens_out)
            
            if eos_detected:
                tail_frames += 1
                if tail_frames >= MAX_TAIL:
                    break
    
final_moshi_codes = torch.cat(collected_moshi_codes, dim=-1) 

print("\n" + "="*40)
print("MOSHI'S TEXT RESPONSE:")
print("".join(generated_text_pieces).strip() or "(No text generated)")
print("="*40 + "\n")

# ==========================================
# 4. PHASE 2: BRIDGING & DSP POST-PROCESSING
# ==========================================
print("🌉 Phase 2: Applying TCN Bridge and DSP Filters...")

bridge = AcousticDisentanglementBridge().to(DEVICE_HOME)
raw_state_dict = torch.load(BRIDGE_PATH, map_location=DEVICE_HOME, weights_only=True)
bridge.load_state_dict(raw_state_dict)
bridge.eval()

sample_data = torch.load(TARGET_PT_PATH, weights_only=True)
id_vec = sample_data["identity_vector"].view(1, -1).to(DEVICE_HOME, dtype=torch.float32)

with torch.no_grad():
    cb0_sequence = final_moshi_codes[:, 0, :] 
    new_logits = bridge(cb0_sequence, id_vec) 
    new_cb1_7 = torch.argmax(new_logits, dim=-1).transpose(1, 2) 
    
    hybrid_codes = final_moshi_codes.clone()
    hybrid_codes[:, 1:8, :] = new_cb1_7
    bridged_waveform = mimi.decode(hybrid_codes).to(torch.float32).cpu()

clean_waveform = torchaudio.functional.lowpass_biquad(bridged_waveform, sample_rate=24000, cutoff_freq=7500.0)

fade_samples = int(24000 * 0.2) 
if clean_waveform.shape[-1] > fade_samples:
    fade_curve = torch.linspace(1.0, 0.0, fade_samples) ** 2 
    clean_waveform[0, 0, -fade_samples:] *= fade_curve

torchaudio.save(OUTPUT_FILENAME, clean_waveform.squeeze(0), 24000)
print(f"🎉 Success! TCN Bridged audio saved to: {OUTPUT_FILENAME}")
display(Audio(OUTPUT_FILENAME, rate=24000))

# ==========================================
# 5. PHASE 3: RESEARCH METRICS REPORT
# ==========================================
print("\n" + "="*50)
print("📊 FINAL RESEARCH METRICS REPORT")
print("="*50)

def get_prosody_stats(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    f0 = f0[~np.isnan(f0)]
    rms = librosa.feature.rms(y=y)[0]
    return {
        "mean_f0": np.mean(f0) if len(f0) > 0 else 0,
        "std_f0": np.std(f0) if len(f0) > 0 else 0,
        "mean_energy": np.mean(rms)
    }

target_stats = get_prosody_stats(REFERENCE_WAV)
bridge_stats = get_prosody_stats(OUTPUT_FILENAME)

print("1️⃣ PROSODY & EMPATHY ALIGNMENT")
print("-" * 45)
print(f"{'Metric':<18} | {'Target (Sad)':<12} | {'Bridge Out':<12}")
print("-" * 45)
print(f"{'Mean Pitch (Hz)':<18} | {target_stats['mean_f0']:>12.2f} | {bridge_stats['mean_f0']:>12.2f}")
print(f"{'Pitch StdDev':<18} | {target_stats['std_f0']:>12.2f} | {bridge_stats['std_f0']:>12.2f}")
print(f"{'Mean Energy':<18} | {target_stats['mean_energy']:>12.6f} | {bridge_stats['mean_energy']:>12.6f}")
print("-" * 45)

try:
    from pymcd.mcd import Calculate_MCD
    mcd_toolbox = Calculate_MCD(MCD_mode="dtw")
    mcd_score = mcd_toolbox.calculate_mcd(REFERENCE_WAV, OUTPUT_FILENAME)
    print(f"\n2️⃣ ACOUSTIC DISTORTION")
    print("-" * 45)
    print(f"📉 Official DTW-MCD Score:     {mcd_score:.2f} dB")
except ImportError:
    print("📉 Please run `!pip install pymcd` to view MCD score.")

print(f"\n3️⃣ BIOMETRIC IDENTITY TRANSFER")
print("-" * 45)
try:
    verification = SpeakerRecognition.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb", 
        run_opts={"device": DEVICE_HOME}
    )
    wav_target = verification.load_audio(REFERENCE_WAV)
    wav_output = verification.load_audio(OUTPUT_FILENAME)
    
    emb1 = verification.encode_batch(wav_target)
    emb2 = verification.encode_batch(wav_output)
    
    similarity = torch.nn.functional.cosine_similarity(emb1.flatten(), emb2.flatten(), dim=0).item()
    print(f"🧬 ECAPA-TDNN Similarity:      {similarity:.4f}")
except Exception as e:
    print(f"⚠️ Identity Check Failed: {e}")

In [ ]:
import os
import torch
import torch.nn as nn
import torchaudio
import sentencepiece
import librosa
import numpy as np
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from speechbrain.inference.speaker import SpeakerRecognition
from torchaudio.models import Conformer


# ==========================================
# 0. THE DYNAMIC GRAPH KILLER
# ==========================================
import moshi.utils.compile

print("🔍 Hunting for Kyutai's CUDA Graph compiler...")
patched_any = False
for name, obj in vars(moshi.utils.compile).items():
    if isinstance(obj, type) and hasattr(obj, '__call__'):
        def make_bypass(orig_call):
            def bypass_call(self, *args, **kwargs):
                if hasattr(self, 'func'): return self.func(*args, **kwargs)
                return orig_call(self, *args, **kwargs)
            return bypass_call
        obj.__call__ = make_bypass(obj.__call__)
        print(f"✅ Successfully neutralized CUDA Graphs in: moshi.utils.compile.{name}")
        patched_any = True

# ==========================================
# 1. SETUP, PATHS & V2.2 BRIDGE ARCHITECTURE
# ==========================================
DEVICE_HOME = "cuda:0" 
DEVICE_WORK = "cuda:1" 

BRIDGE_PATH = "/kaggle/input/datasets/ahmedsadman099876/conformer/acoustic_bridge_best_conformer__1.pt" 
TARGET_PT_PATH = "/kaggle/input/datasets/ahmedsadman099876/data899/0012_001164.pt"
USER_PROMPT_WAV = "/kaggle/input/datasets/ahmedsadman099876/recording/Recording.wav"
REFERENCE_WAV = "/kaggle/input/datasets/ahmedsadman099876/data458/0012_001164.wav"
OUTPUT_FILENAME = "/kaggle/working/final_bridged_eval_DSP_Tuned.wav"

class AcousticDisentanglementBridge(nn.Module):
    def __init__(self, vocab_size=2048, embed_dim=1024, id_dim=192, max_len=4096):
        super().__init__()
        self.cb0_embedding = nn.Embedding(vocab_size, embed_dim)
        self.id_projection = nn.Linear(id_dim, embed_dim)
        self.positional_encoding = nn.Embedding(max_len, embed_dim)
        
        # 🚀 UPGRADE: The Unified Conformer Block
        # This replaces both the independent cnn_prenet and temporal_transformer
        self.conformer = Conformer(
            input_dim=embed_dim,
            num_heads=8,
            ffn_dim=embed_dim * 4,
            num_layers=2,
            depthwise_conv_kernel_size=15,
            dropout=0.1
        )
        
        self.acoustic_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, embed_dim // 2),
                nn.GELU(),
                nn.LayerNorm(embed_dim // 2),
                nn.Linear(embed_dim // 2, vocab_size)
            ) for _ in range(7)
        ])

    def apply_ppmq_adain(self, cb0_emb, id_vec):
        id_raw = self.id_projection(id_vec).unsqueeze(1)
        id_min, id_max = id_raw.min(dim=-1, keepdim=True)[0], id_raw.max(dim=-1, keepdim=True)[0]
        scale = (id_max - id_min) / 15
        id_quant = torch.round((id_raw - id_min) / (scale + 1e-8)) * scale + id_min
        
        cb0_std, cb0_mean = cb0_emb.std(dim=-1, keepdim=True), cb0_emb.mean(dim=-1, keepdim=True)
        id_std, id_mean = id_quant.std(dim=-1, keepdim=True) + 1e-8, id_quant.mean(dim=-1, keepdim=True)
        
        
        return cb0_emb + ((cb0_std * ((id_quant - id_mean) / id_std) + cb0_mean) * 0.7)

    def forward(self, cb0_tokens, id_vec):
        if cb0_tokens.dim() == 3: cb0_tokens = cb0_tokens.view(cb0_tokens.shape[0], -1)
        seq_len = cb0_tokens.size(1)
        device = cb0_tokens.device
        
        positions = torch.arange(0, seq_len, device=device).unsqueeze(0)
        cb0_emb = self.cb0_embedding(cb0_tokens) + self.positional_encoding(positions)
        
        fused_emb = self.apply_ppmq_adain(cb0_emb, id_vec)
        fused_emb = torch.nan_to_num(fused_emb, nan=0.0, posinf=1.0, neginf=-1.0)
        
        # 🚀 CONFORMER PASS
        # The Conformer requires a 'lengths' tensor to know exactly how much sequence to process
        lengths = torch.tensor([seq_len] * fused_emb.shape[0], device=device)
        temporal_features, _ = self.conformer(fused_emb, lengths)
        
        logits_list = [head(temporal_features) for head in self.acoustic_heads]
        return torch.stack(logits_list, dim=2)

# ==========================================
# 2. LOAD MODELS & CONSTRUCT SANDWICH
# ==========================================
from moshi.models import loaders, LMGen

repo_id = "kyutai/moshika-pytorch-bf16"
print("\nDownloading Mimi weights...")
mimi = loaders.get_mimi(hf_hub_download(repo_id, "tokenizer-e351c8d8-checkpoint125.safetensors"), device="cpu")
mimi = mimi.to(DEVICE_HOME)
mimi.set_num_codebooks(8)

print("Downloading Moshi weights (~15.4 GB)...")
moshi = loaders.get_moshi_lm(hf_hub_download(repo_id, "model.safetensors"), device="cpu")
text_tokenizer = sentencepiece.SentencePieceProcessor(hf_hub_download(repo_id, "tokenizer_spm_32k_3.model"))

print("\n🛠️ Sharding Model across GPUs...")
def shard_hook(module, args):
    try: target_dev = next(module.parameters()).device
    except StopIteration:
        try: target_dev = next(module.buffers()).device
        except StopIteration: return args
    if isinstance(args, tuple): return tuple(a.to(target_dev) if isinstance(a, torch.Tensor) else a for a in args)
    return args.to(target_dev) if isinstance(args, torch.Tensor) else args

mid = len(moshi.transformer.layers) // 2
for i in range(mid, len(moshi.transformer.layers)):
    moshi.transformer.layers[i].to(DEVICE_WORK)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)

moshi.emb.to(DEVICE_HOME)
moshi.text_emb.to(DEVICE_HOME)
for i in range(mid):
    moshi.transformer.layers[i].to(DEVICE_HOME)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)
if hasattr(moshi.transformer, "norm"):
    moshi.transformer.norm.to(DEVICE_HOME)
    moshi.transformer.norm.register_forward_pre_hook(shard_hook)
for name, module in moshi.named_children():
    if name not in ["emb", "text_emb", "transformer"]:
        module.to(DEVICE_HOME)
        module.register_forward_pre_hook(shard_hook)

for name, buf in moshi.named_buffers(recurse=True): buf.data = buf.data.to(DEVICE_HOME)
for attr in ['initial', 'delays', 'zero_token_id']:
    if hasattr(moshi, attr):
        t = getattr(moshi, attr)
        if isinstance(t, torch.Tensor): setattr(moshi, attr, t.to(DEVICE_HOME))

lm_gen = LMGen(moshi, temp=0.8, temp_text=0.7)

# ==========================================
# 3. PHASE 1: NATIVE GENERATION (SMART TRIMMER)
# ==========================================
try:
    wav, sr = torchaudio.load(USER_PROMPT_WAV)
    if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
    if sr != 24000: wav = torchaudio.functional.resample(wav, sr, 24000)
    wav = wav.unsqueeze(0).to(DEVICE_HOME)
except FileNotFoundError:
    wav = torch.zeros(1, 1, 24000 * 3).to(DEVICE_HOME)

print("\n⚡ Phase 1: Generating Base Tokens Natively...")
frame_size = mimi.frame_size
all_codes = []
with torch.no_grad(), mimi.streaming(batch_size=1):
    for offset in range(0, wav.shape[-1], frame_size):
        frame = wav[:, :, offset : offset + frame_size]
        if frame.shape[-1] < frame_size: frame = torch.nn.functional.pad(frame, (0, frame_size - frame.shape[-1]))
        all_codes.append(mimi.encode(frame))

collected_moshi_codes = []
generated_text_pieces = []

with torch.no_grad(), lm_gen.streaming(1), mimi.streaming(1):
    def process_step(tokens_out):
        text_token = tokens_out[0, 0].item()
        if text_token not in (0, 3): 
            generated_text_pieces.append(text_tokenizer.id_to_piece(text_token).replace(' ', ' '))
        collected_moshi_codes.append(tokens_out[:, 1:])

    for code in all_codes:
        tokens_out = lm_gen.step(code)
        if tokens_out is not None: process_step(tokens_out)

    print("Moshi is replying...")
    silence_code = torch.zeros_like(all_codes[0]).to(DEVICE_HOME)
    
    eos_detected = False
    tail_frames = 0
    MAX_TAIL = 4  
    
    for _ in range(120): 
        tokens_out = lm_gen.step(silence_code)
        if tokens_out is not None: 
            text_token = tokens_out[0, 0].item()
            if text_token in (0, 3):
                eos_detected = True
            process_step(tokens_out)
            
            if eos_detected:
                tail_frames += 1
                if tail_frames >= MAX_TAIL:
                    print(f"   [Sentence complete. Cutting after {MAX_TAIL} frames to prevent static.]")
                    break
    
final_moshi_codes = torch.cat(collected_moshi_codes, dim=-1) 

print("\n" + "="*40)
print("MOSHI'S TEXT RESPONSE:")
print("".join(generated_text_pieces).strip() or "(No text generated)")
print("="*40 + "\n")

# ==========================================
# 4. PHASE 2: BRIDGING & DSP POST-PROCESSING
# ==========================================
print("🌉 Phase 2: Applying Bridge and DSP Filters...")

bridge = AcousticDisentanglementBridge().to(DEVICE_HOME)

# 🛠️ THE FIX: Torchaudio Version Translator
raw_state_dict = torch.load(BRIDGE_PATH, map_location=DEVICE_HOME, weights_only=True)
translated_state_dict = {}

for k, v in raw_state_dict.items():
    # 1. Add the missing parent prefix
    if k.startswith("conformer_layers."):
        k = "conformer." + k
        
    # 2. Translate old layer names to the new Sequential format
    k = k.replace("ffn1.ln.", "ffn1.sequential.0.")
    k = k.replace("ffn1.linear1.", "ffn1.sequential.1.")
    k = k.replace("ffn1.linear2.", "ffn1.sequential.4.")
    
    k = k.replace("ffn2.ln.", "ffn2.sequential.0.")
    k = k.replace("ffn2.linear1.", "ffn2.sequential.1.")
    k = k.replace("ffn2.linear2.", "ffn2.sequential.4.")
    
    k = k.replace("ln_mha.", "self_attn_layer_norm.")
    k = k.replace("mha.", "self_attn.")
    
    k = k.replace("conv_module.ln.", "conv_module.layer_norm.")
    k = k.replace("conv_module.pointwise1.", "conv_module.sequential.0.")
    k = k.replace("conv_module.depthwise.", "conv_module.sequential.2.")
    k = k.replace("conv_module.bn.", "conv_module.sequential.3.")
    k = k.replace("conv_module.pointwise2.", "conv_module.sequential.5.")
    
    k = k.replace("final_ln.", "final_layer_norm.")
    
    translated_state_dict[k] = v

# Load the freshly translated weights!
bridge.load_state_dict(translated_state_dict)
bridge.eval()

sample_data = torch.load(TARGET_PT_PATH, weights_only=True)
id_vec = sample_data["identity_vector"].view(1, -1).to(DEVICE_HOME, dtype=torch.float32)

with torch.no_grad():
    cb0_sequence = final_moshi_codes[:, 0, :] 
    new_logits = bridge(cb0_sequence, id_vec) 
    new_cb1_7 = torch.argmax(new_logits, dim=-1).transpose(1, 2) 
    
    hybrid_codes = final_moshi_codes.clone()
    hybrid_codes[:, 1:8, :] = new_cb1_7
    bridged_waveform = mimi.decode(hybrid_codes).to(torch.float32).cpu()

clean_waveform = torchaudio.functional.lowpass_biquad(bridged_waveform, sample_rate=24000, cutoff_freq=7500.0)

fade_samples = int(24000 * 0.2) 
if clean_waveform.shape[-1] > fade_samples:
    fade_curve = torch.linspace(1.0, 0.0, fade_samples) ** 2 
    clean_waveform[0, 0, -fade_samples:] *= fade_curve

torchaudio.save(OUTPUT_FILENAME, clean_waveform.squeeze(0), 24000)
print(f"🎉 Success! Bridged audio saved to: {OUTPUT_FILENAME}")
display(Audio(OUTPUT_FILENAME, rate=24000))

# ==========================================
# 5. PHASE 3: RESEARCH METRICS REPORT
# ==========================================
print("\n" + "="*50)
print("📊 FINAL RESEARCH METRICS REPORT")
print("="*50)

# --- A. OBJECTIVE PROSODY (PITCH & ENERGY) ---
def get_prosody_stats(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    f0 = f0[~np.isnan(f0)]
    rms = librosa.feature.rms(y=y)[0]
    return {
        "mean_f0": np.mean(f0) if len(f0) > 0 else 0,
        "std_f0": np.std(f0) if len(f0) > 0 else 0,
        "mean_energy": np.mean(rms)
    }

target_stats = get_prosody_stats(REFERENCE_WAV)
bridge_stats = get_prosody_stats(OUTPUT_FILENAME)

print("1️⃣ PROSODY & EMPATHY ALIGNMENT")
print("-" * 45)
print(f"{'Metric':<18} | {'Target (Sad)':<12} | {'Bridge Out':<12}")
print("-" * 45)
print(f"{'Mean Pitch (Hz)':<18} | {target_stats['mean_f0']:>12.2f} | {bridge_stats['mean_f0']:>12.2f}")
print(f"{'Pitch StdDev':<18} | {target_stats['std_f0']:>12.2f} | {bridge_stats['std_f0']:>12.2f}")
print(f"{'Mean Energy':<18} | {target_stats['mean_energy']:>12.6f} | {bridge_stats['mean_energy']:>12.6f}")
print("-" * 45)

from pymcd.mcd import Calculate_MCD

# Initialize the official metric calculator
# "dtw" mode automatically aligns the audio before scoring
mcd_toolbox = Calculate_MCD(MCD_mode="dtw")

# Calculate the true MCD
mcd_score = mcd_toolbox.calculate_mcd(REFERENCE_WAV, OUTPUT_FILENAME)

print(f"\n2️⃣ ACOUSTIC DISTORTION")
print("-" * 45)
print(f"📉 Official DTW-MCD Score:     {mcd_score:.2f} dB")

# --- C. ECAPA-TDNN IDENTITY CHECK ---
print(f"\n3️⃣ BIOMETRIC IDENTITY TRANSFER")
print("-" * 45)
try:
    verification = SpeakerRecognition.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb", 
        run_opts={"device": DEVICE_HOME}
    )
    wav_target = verification.load_audio(REFERENCE_WAV)
    wav_output = verification.load_audio(OUTPUT_FILENAME)
    
    emb1 = verification.encode_batch(wav_target)
    emb2 = verification.encode_batch(wav_output)
    
    similarity = torch.nn.functional.cosine_similarity(emb1.flatten(), emb2.flatten(), dim=0).item()
    print(f"🧬 ECAPA-TDNN Similarity:      {similarity:.4f}")
except Exception as e:
    print(f"⚠️ Identity Check Failed: {e}")

print("="*50)

In [ ]:
import os
import torch
import torch.nn as nn
import torchaudio
import sentencepiece
import librosa
import numpy as np
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from speechbrain.inference.speaker import SpeakerRecognition


# ==========================================
# 0. THE DYNAMIC GRAPH KILLER
# ==========================================
import moshi.utils.compile

print("🔍 Hunting for Kyutai's CUDA Graph compiler...")
patched_any = False
for name, obj in vars(moshi.utils.compile).items():
    if isinstance(obj, type) and hasattr(obj, '__call__'):
        def make_bypass(orig_call):
            def bypass_call(self, *args, **kwargs):
                if hasattr(self, 'func'): return self.func(*args, **kwargs)
                return orig_call(self, *args, **kwargs)
            return bypass_call
        obj.__call__ = make_bypass(obj.__call__)
        print(f"✅ Successfully neutralized CUDA Graphs in: moshi.utils.compile.{name}")
        patched_any = True

# ==========================================
# 1. SETUP, PATHS & V2.2 BRIDGE ARCHITECTURE
# ==========================================
DEVICE_HOME = "cuda:0" 
DEVICE_WORK = "cuda:1" 

BRIDGE_PATH = "/kaggle/input/datasets/ahmedsadman099876/data999/acoustic_bridge_best__1.pt" 
TARGET_PT_PATH = "/kaggle/input/datasets/ahmedsadman099876/data899/0012_001164.pt"
USER_PROMPT_WAV = "/kaggle/input/datasets/ahmedsadman099876/recording/Recording.wav"
REFERENCE_WAV = "/kaggle/input/datasets/ahmedsadman099876/data458/0012_001164.wav"
OUTPUT_FILENAME = "/kaggle/working/final_bridged_eval_DSP_Tuned.wav"

class AcousticDisentanglementBridge(nn.Module):
    def __init__(self, vocab_size=2048, embed_dim=1024, id_dim=192, max_len=4096):
        super().__init__()
        self.cb0_embedding = nn.Embedding(vocab_size, embed_dim)
        self.id_projection = nn.Linear(id_dim, embed_dim)
        self.positional_encoding = nn.Embedding(max_len, embed_dim)
        
        self.cnn_prenet = nn.Sequential(
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            nn.GELU()
        )
        
        self.temporal_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, batch_first=True, dropout=0.1), 
            num_layers=2
        )
        
        self.acoustic_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, embed_dim // 2),
                nn.GELU(),
                nn.LayerNorm(embed_dim // 2),
                nn.Linear(embed_dim // 2, vocab_size)
            ) for _ in range(7)
        ])

    def apply_ppmq_adain(self, cb0_emb, id_vec):
        id_raw = self.id_projection(id_vec).unsqueeze(1)
        id_min, id_max = id_raw.min(dim=-1, keepdim=True)[0], id_raw.max(dim=-1, keepdim=True)[0]
        scale = (id_max - id_min) / 15
        id_quant = torch.round((id_raw - id_min) / (scale + 1e-8)) * scale + id_min
        
        cb0_std, cb0_mean = cb0_emb.std(dim=-1, keepdim=True), cb0_emb.mean(dim=-1, keepdim=True)
        id_std, id_mean = id_quant.std(dim=-1, keepdim=True) + 1e-8, id_quant.mean(dim=-1, keepdim=True)
        
        return cb0_emb + ((cb0_std * ((id_quant - id_mean) / id_std) + cb0_mean) * 1)

    def forward(self, cb0_tokens, id_vec):
        if cb0_tokens.dim() == 3: cb0_tokens = cb0_tokens.view(cb0_tokens.shape[0], -1)
        seq_len = cb0_tokens.size(1)
        device = cb0_tokens.device
        
        positions = torch.arange(0, seq_len, device=device).unsqueeze(0)
        cb0_emb = self.cb0_embedding(cb0_tokens) + self.positional_encoding(positions)
        
        fused_emb = self.apply_ppmq_adain(cb0_emb, id_vec)
        fused_emb = torch.nan_to_num(fused_emb, nan=0.0, posinf=1.0, neginf=-1.0)
        
        fused_emb = fused_emb.transpose(1, 2)
        fused_emb = self.cnn_prenet(fused_emb)
        fused_emb = fused_emb.transpose(1, 2)
        
        temporal_features = self.temporal_transformer(fused_emb)
        logits_list = [head(temporal_features) for head in self.acoustic_heads]
        return torch.stack(logits_list, dim=2)

# ==========================================
# 2. LOAD MODELS & CONSTRUCT SANDWICH
# ==========================================
from moshi.models import loaders, LMGen

repo_id = "kyutai/moshika-pytorch-bf16"
print("\nDownloading Mimi weights...")
mimi = loaders.get_mimi(hf_hub_download(repo_id, "tokenizer-e351c8d8-checkpoint125.safetensors"), device="cpu")
mimi = mimi.to(DEVICE_HOME)
mimi.set_num_codebooks(8)

print("Downloading Moshi weights (~15.4 GB)...")
moshi = loaders.get_moshi_lm(hf_hub_download(repo_id, "model.safetensors"), device="cpu")
text_tokenizer = sentencepiece.SentencePieceProcessor(hf_hub_download(repo_id, "tokenizer_spm_32k_3.model"))

print("\n🛠️ Sharding Model across GPUs...")
def shard_hook(module, args):
    try: target_dev = next(module.parameters()).device
    except StopIteration:
        try: target_dev = next(module.buffers()).device
        except StopIteration: return args
    if isinstance(args, tuple): return tuple(a.to(target_dev) if isinstance(a, torch.Tensor) else a for a in args)
    return args.to(target_dev) if isinstance(args, torch.Tensor) else args

mid = len(moshi.transformer.layers) // 2
for i in range(mid, len(moshi.transformer.layers)):
    moshi.transformer.layers[i].to(DEVICE_WORK)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)

moshi.emb.to(DEVICE_HOME)
moshi.text_emb.to(DEVICE_HOME)
for i in range(mid):
    moshi.transformer.layers[i].to(DEVICE_HOME)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)
if hasattr(moshi.transformer, "norm"):
    moshi.transformer.norm.to(DEVICE_HOME)
    moshi.transformer.norm.register_forward_pre_hook(shard_hook)
for name, module in moshi.named_children():
    if name not in ["emb", "text_emb", "transformer"]:
        module.to(DEVICE_HOME)
        module.register_forward_pre_hook(shard_hook)

for name, buf in moshi.named_buffers(recurse=True): buf.data = buf.data.to(DEVICE_HOME)
for attr in ['initial', 'delays', 'zero_token_id']:
    if hasattr(moshi, attr):
        t = getattr(moshi, attr)
        if isinstance(t, torch.Tensor): setattr(moshi, attr, t.to(DEVICE_HOME))

lm_gen = LMGen(moshi, temp=0.8, temp_text=0.7)

# ==========================================
# 3. PHASE 1: NATIVE GENERATION (SMART TRIMMER)
# ==========================================
try:
    wav, sr = torchaudio.load(USER_PROMPT_WAV)
    if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
    if sr != 24000: wav = torchaudio.functional.resample(wav, sr, 24000)
    wav = wav.unsqueeze(0).to(DEVICE_HOME)
except FileNotFoundError:
    wav = torch.zeros(1, 1, 24000 * 3).to(DEVICE_HOME)

print("\n⚡ Phase 1: Generating Base Tokens Natively...")
frame_size = mimi.frame_size
all_codes = []
with torch.no_grad(), mimi.streaming(batch_size=1):
    for offset in range(0, wav.shape[-1], frame_size):
        frame = wav[:, :, offset : offset + frame_size]
        if frame.shape[-1] < frame_size: frame = torch.nn.functional.pad(frame, (0, frame_size - frame.shape[-1]))
        all_codes.append(mimi.encode(frame))

collected_moshi_codes = []
generated_text_pieces = []

with torch.no_grad(), lm_gen.streaming(1), mimi.streaming(1):
    def process_step(tokens_out):
        text_token = tokens_out[0, 0].item()
        if text_token not in (0, 3): 
            generated_text_pieces.append(text_tokenizer.id_to_piece(text_token).replace(' ', ' '))
        collected_moshi_codes.append(tokens_out[:, 1:])

    for code in all_codes:
        tokens_out = lm_gen.step(code)
        if tokens_out is not None: process_step(tokens_out)

    print("Moshi is replying...")
    silence_code = torch.zeros_like(all_codes[0]).to(DEVICE_HOME)
    
    eos_detected = False
    tail_frames = 0
    MAX_TAIL = 4  
    
    for _ in range(120): 
        tokens_out = lm_gen.step(silence_code)
        if tokens_out is not None: 
            text_token = tokens_out[0, 0].item()
            if text_token in (0, 3):
                eos_detected = True
            process_step(tokens_out)
            
            if eos_detected:
                tail_frames += 1
                if tail_frames >= MAX_TAIL:
                    print(f"   [Sentence complete. Cutting after {MAX_TAIL} frames to prevent static.]")
                    break
    
final_moshi_codes = torch.cat(collected_moshi_codes, dim=-1) 

print("\n" + "="*40)
print("MOSHI'S TEXT RESPONSE:")
print("".join(generated_text_pieces).strip() or "(No text generated)")
print("="*40 + "\n")

# ==========================================
# 4. PHASE 2: BRIDGING & DSP POST-PROCESSING
# ==========================================
print("🌉 Phase 2: Applying Bridge and DSP Filters...")

bridge = AcousticDisentanglementBridge().to(DEVICE_HOME)
bridge.load_state_dict(torch.load(BRIDGE_PATH, map_location=DEVICE_HOME, weights_only=True))
bridge.eval()

sample_data = torch.load(TARGET_PT_PATH, weights_only=True)
id_vec = sample_data["identity_vector"].view(1, -1).to(DEVICE_HOME, dtype=torch.float32)

with torch.no_grad():
    cb0_sequence = final_moshi_codes[:, 0, :] 
    new_logits = bridge(cb0_sequence, id_vec) 
    new_cb1_7 = torch.argmax(new_logits, dim=-1).transpose(1, 2) 
    
    hybrid_codes = final_moshi_codes.clone()
    hybrid_codes[:, 1:8, :] = new_cb1_7
    bridged_waveform = mimi.decode(hybrid_codes).to(torch.float32).cpu()

clean_waveform = torchaudio.functional.lowpass_biquad(bridged_waveform, sample_rate=24000, cutoff_freq=7500.0)

fade_samples = int(24000 * 0.2) 
if clean_waveform.shape[-1] > fade_samples:
    fade_curve = torch.linspace(1.0, 0.0, fade_samples) ** 2 
    clean_waveform[0, 0, -fade_samples:] *= fade_curve

torchaudio.save(OUTPUT_FILENAME, clean_waveform.squeeze(0), 24000)
print(f"🎉 Success! Bridged audio saved to: {OUTPUT_FILENAME}")
display(Audio(OUTPUT_FILENAME, rate=24000))

# ==========================================
# 5. PHASE 3: RESEARCH METRICS REPORT
# ==========================================
print("\n" + "="*50)
print("📊 FINAL RESEARCH METRICS REPORT")
print("="*50)

# --- A. OBJECTIVE PROSODY (PITCH & ENERGY) ---
def get_prosody_stats(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    f0 = f0[~np.isnan(f0)]
    rms = librosa.feature.rms(y=y)[0]
    return {
        "mean_f0": np.mean(f0) if len(f0) > 0 else 0,
        "std_f0": np.std(f0) if len(f0) > 0 else 0,
        "mean_energy": np.mean(rms)
    }

target_stats = get_prosody_stats(REFERENCE_WAV)
bridge_stats = get_prosody_stats(OUTPUT_FILENAME)

print("1️⃣ PROSODY & EMPATHY ALIGNMENT")
print("-" * 45)
print(f"{'Metric':<18} | {'Target (Sad)':<12} | {'Bridge Out':<12}")
print("-" * 45)
print(f"{'Mean Pitch (Hz)':<18} | {target_stats['mean_f0']:>12.2f} | {bridge_stats['mean_f0']:>12.2f}")
print(f"{'Pitch StdDev':<18} | {target_stats['std_f0']:>12.2f} | {bridge_stats['std_f0']:>12.2f}")
print(f"{'Mean Energy':<18} | {target_stats['mean_energy']:>12.6f} | {bridge_stats['mean_energy']:>12.6f}")
print("-" * 45)

from pymcd.mcd import Calculate_MCD

# Initialize the official metric calculator
# "dtw" mode automatically aligns the audio before scoring
mcd_toolbox = Calculate_MCD(MCD_mode="dtw")

# Calculate the true MCD
mcd_score = mcd_toolbox.calculate_mcd(REFERENCE_WAV, OUTPUT_FILENAME)

print(f"\n2️⃣ ACOUSTIC DISTORTION")
print("-" * 45)
print(f"📉 Official DTW-MCD Score:     {mcd_score:.2f} dB")

# --- C. ECAPA-TDNN IDENTITY CHECK ---
print(f"\n3️⃣ BIOMETRIC IDENTITY TRANSFER")
print("-" * 45)
try:
    verification = SpeakerRecognition.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb", 
        run_opts={"device": DEVICE_HOME}
    )
    wav_target = verification.load_audio(REFERENCE_WAV)
    wav_output = verification.load_audio(OUTPUT_FILENAME)
    
    emb1 = verification.encode_batch(wav_target)
    emb2 = verification.encode_batch(wav_output)
    
    similarity = torch.nn.functional.cosine_similarity(emb1.flatten(), emb2.flatten(), dim=0).item()
    print(f"🧬 ECAPA-TDNN Similarity:      {similarity:.4f}")
except Exception as e:
    print(f"⚠️ Identity Check Failed: {e}")

print("="*50)

In [ ]:
import torch
import torch.nn as nn
import torchaudio
import sentencepiece
import librosa
import numpy as np
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from speechbrain.inference.speaker import SpeakerRecognition

# ==========================================
# 0. THE DYNAMIC GRAPH KILLER
# ==========================================
import moshi.utils.compile

print("🔍 Hunting for Kyutai's CUDA Graph compiler...")
patched_any = False
for name, obj in vars(moshi.utils.compile).items():
    if isinstance(obj, type) and hasattr(obj, '__call__'):
        def make_bypass(orig_call):
            def bypass_call(self, *args, **kwargs):
                if hasattr(self, 'func'): return self.func(*args, **kwargs)
                return orig_call(self, *args, **kwargs)
            return bypass_call
        obj.__call__ = make_bypass(obj.__call__)
        print(f"✅ Successfully neutralized CUDA Graphs in: moshi.utils.compile.{name}")
        patched_any = True

# ==========================================
# 1. SETUP, PATHS & BRIDGE ARCHITECTURE
# ==========================================
DEVICE_HOME = "cuda:0" 
DEVICE_WORK = "cuda:1" 

BRIDGE_PATH = "/kaggle/input/datasets/ahmedsadman099876/data589/acoustic_bridge_final_bf16_aligned.pt"
TARGET_PT_PATH = "/kaggle/input/datasets/ahmedsadman099876/data899/0012_001164.pt"
USER_PROMPT_WAV = "/kaggle/input/datasets/ahmedsadman099876/recording/Recording.wav"
REFERENCE_WAV = "/kaggle/input/datasets/ahmedsadman099876/data458/0012_001164.wav"

class AcousticDisentanglementBridge(nn.Module):
    def __init__(self, vocab_size=2048, embed_dim=1024, id_dim=192):
        super().__init__()
        self.cb0_embedding = nn.Embedding(vocab_size, embed_dim)
        self.id_projection = nn.Linear(id_dim, embed_dim)
        self.temporal_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, batch_first=True), 
            num_layers=2
        )
        self.acoustic_heads = nn.ModuleList([nn.Linear(embed_dim, vocab_size) for _ in range(7)])

    def apply_ppmq_adain(self, cb0_emb, id_vec):
        id_raw = self.id_projection(id_vec).unsqueeze(1)
        id_min, id_max = id_raw.min(dim=-1, keepdim=True)[0], id_raw.max(dim=-1, keepdim=True)[0]
        scale = (id_max - id_min) / 15
        id_quant = torch.round((id_raw - id_min) / (scale + 1e-8)) * scale + id_min
        cb0_std, cb0_mean = cb0_emb.std(dim=-1, keepdim=True), cb0_emb.mean(dim=-1, keepdim=True)
        id_std, id_mean = id_quant.std(dim=-1, keepdim=True) + 1e-8, id_quant.mean(dim=-1, keepdim=True)
        return cb0_emb + ((cb0_std * ((id_quant - id_mean) / id_std) + cb0_mean) * 0.8)

    def forward(self, cb0_tokens, id_vec):
        if cb0_tokens.dim() == 3: cb0_tokens = cb0_tokens.view(cb0_tokens.shape[0], -1)
        cb0_emb = self.cb0_embedding(cb0_tokens)
        fused_emb = self.apply_ppmq_adain(cb0_emb, id_vec)
        # Safe-catch for AdaIN NaNs
        fused_emb = torch.nan_to_num(fused_emb, nan=0.0, posinf=1.0, neginf=-1.0)
        temporal_features = self.temporal_transformer(fused_emb)
        logits_list = [head(temporal_features) for head in self.acoustic_heads]
        return torch.stack(logits_list, dim=2)

# ==========================================
# 2. LOAD MODELS & CONSTRUCT SANDWICH
# ==========================================
from moshi.models import loaders, LMGen

repo_id = "kyutai/moshika-pytorch-bf16"
print("\nDownloading Mimi weights...")
mimi = loaders.get_mimi(hf_hub_download(repo_id, "tokenizer-e351c8d8-checkpoint125.safetensors"), device="cpu")
mimi = mimi.to(DEVICE_HOME)
mimi.set_num_codebooks(8)

print("Downloading Moshi weights (~15.4 GB)...")
moshi = loaders.get_moshi_lm(hf_hub_download(repo_id, "model.safetensors"), device="cpu")
text_tokenizer = sentencepiece.SentencePieceProcessor(hf_hub_download(repo_id, "tokenizer_spm_32k_3.model"))

print("\n🛠️ Sharding Model across GPUs...")
def shard_hook(module, args):
    try: target_dev = next(module.parameters()).device
    except StopIteration:
        try: target_dev = next(module.buffers()).device
        except StopIteration: return args
    if isinstance(args, tuple): return tuple(a.to(target_dev) if isinstance(a, torch.Tensor) else a for a in args)
    return args.to(target_dev) if isinstance(args, torch.Tensor) else args

mid = len(moshi.transformer.layers) // 2
for i in range(mid, len(moshi.transformer.layers)):
    moshi.transformer.layers[i].to(DEVICE_WORK)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)

moshi.emb.to(DEVICE_HOME)
moshi.text_emb.to(DEVICE_HOME)
for i in range(mid):
    moshi.transformer.layers[i].to(DEVICE_HOME)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)
if hasattr(moshi.transformer, "norm"):
    moshi.transformer.norm.to(DEVICE_HOME)
    moshi.transformer.norm.register_forward_pre_hook(shard_hook)
for name, module in moshi.named_children():
    if name not in ["emb", "text_emb", "transformer"]:
        module.to(DEVICE_HOME)
        module.register_forward_pre_hook(shard_hook)

for name, buf in moshi.named_buffers(recurse=True): buf.data = buf.data.to(DEVICE_HOME)
for attr in ['initial', 'delays', 'zero_token_id']:
    if hasattr(moshi, attr):
        t = getattr(moshi, attr)
        if isinstance(t, torch.Tensor): setattr(moshi, attr, t.to(DEVICE_HOME))

lm_gen = LMGen(moshi, temp=0.8, temp_text=0.7)

# ==========================================
# 3. PHASE 1: NATIVE GENERATION
# ==========================================
try:
    wav, sr = torchaudio.load(USER_PROMPT_WAV)
    if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
    if sr != 24000: wav = torchaudio.functional.resample(wav, sr, 24000)
    wav = wav.unsqueeze(0).to(DEVICE_HOME)
except FileNotFoundError:
    wav = torch.zeros(1, 1, 24000 * 3).to(DEVICE_HOME)

print("\n⚡ Phase 1: Generating Base Tokens Natively...")
frame_size = mimi.frame_size
all_codes = []
with torch.no_grad(), mimi.streaming(batch_size=1):
    for offset in range(0, wav.shape[-1], frame_size):
        frame = wav[:, :, offset : offset + frame_size]
        if frame.shape[-1] < frame_size: frame = torch.nn.functional.pad(frame, (0, frame_size - frame.shape[-1]))
        all_codes.append(mimi.encode(frame))

collected_moshi_codes = []
generated_text_pieces = []

with torch.no_grad(), lm_gen.streaming(1), mimi.streaming(1):
    def process_step(tokens_out):
        text_token = tokens_out[0, 0].item()
        if text_token not in (0, 3): 
            generated_text_pieces.append(text_tokenizer.id_to_piece(text_token).replace(' ', ' '))
        # CRITICAL SHAPE FIX: Removed .unsqueeze(-1)
        collected_moshi_codes.append(tokens_out[:, 1:])

    for code in all_codes:
        tokens_out = lm_gen.step(code)
        if tokens_out is not None: process_step(tokens_out)

    print("Moshi is replying...")
    silence_code = torch.zeros_like(all_codes[0]).to(DEVICE_HOME)
    
    for _ in range(120):  # Increased range just in case it has a long thought
        tokens_out = lm_gen.step(silence_code)
        if tokens_out is not None: 
            text_token = tokens_out[0, 0].item()
            
            # 🛑 THE SILENCE CUTOFF 🛑
            # If Moshi outputs token 0 or 3, it has finished speaking.
            # We break the loop immediately so we don't feed static to the Bridge!
            if text_token in (0, 3):
                print("   [Moshi finished speaking. Cutting audio to prevent static.]")
                break
                
            process_step(tokens_out)
    

final_moshi_codes = torch.cat(collected_moshi_codes, dim=-1) # Clean Shape: [1, 8, Total_Frames]

print("\n" + "="*40)
print("MOSHI'S TEXT RESPONSE:")
print("".join(generated_text_pieces).strip() or "(No text generated)")
print("="*40 + "\n")

# ==========================================
# 4. PHASE 2: FULL-SEQUENCE BRIDGING
# ==========================================
print("🌉 Phase 2: Applying Acoustic Disentanglement Bridge...")

bridge = AcousticDisentanglementBridge().to(DEVICE_HOME)
bridge.load_state_dict(torch.load(BRIDGE_PATH, map_location=DEVICE_HOME, weights_only=True))
bridge.eval()

sample_data = torch.load(TARGET_PT_PATH, weights_only=True)
# CRITICAL SHAPE FIX: Enforce [1, 192] shape explicitly
id_vec = sample_data["identity_vector"].view(1, -1).to(DEVICE_HOME, dtype=torch.float32)

with torch.no_grad():
    # 1. Extract full prosody sequence
    cb0_sequence = final_moshi_codes[:, 0, :] 
    
    # 2. Bridge temporal forward pass
    new_logits = bridge(cb0_sequence, id_vec) 
    new_cb1_7 = torch.argmax(new_logits, dim=-1).transpose(1, 2) 
    
    # 3. Splice target acoustic codes
    hybrid_codes = final_moshi_codes.clone()
    hybrid_codes[:, 1:8, :] = new_cb1_7
    
    # 4. Decode
    bridged_waveform = mimi.decode(hybrid_codes).to(torch.float32).cpu()

output_filename = "/kaggle/working/final_bridged_eval.wav"
torchaudio.save(output_filename, bridged_waveform.squeeze(0), 24000)

# ==========================================
# 5. PHASE 3: RESEARCH METRICS REPORT
# ==========================================
print("\n" + "="*50)
print("📊 FINAL RESEARCH METRICS REPORT")
print("="*50)

# --- A. OBJECTIVE PROSODY (PITCH & ENERGY) ---
def get_prosody_stats(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    f0 = f0[~np.isnan(f0)]
    rms = librosa.feature.rms(y=y)[0]
    return {
        "mean_f0": np.mean(f0) if len(f0) > 0 else 0,
        "std_f0": np.std(f0) if len(f0) > 0 else 0,
        "mean_energy": np.mean(rms)
    }

target_stats = get_prosody_stats(REFERENCE_WAV)
bridge_stats = get_prosody_stats(output_filename)

print("1️⃣ PROSODY & EMPATHY ALIGNMENT")
print("-" * 45)
print(f"{'Metric':<18} | {'Target (Sad)':<12} | {'Bridge Out':<12}")
print("-" * 45)
print(f"{'Mean Pitch (Hz)':<18} | {target_stats['mean_f0']:>12.2f} | {bridge_stats['mean_f0']:>12.2f}")
print(f"{'Pitch StdDev':<18} | {target_stats['std_f0']:>12.2f} | {bridge_stats['std_f0']:>12.2f}")
print(f"{'Mean Energy':<18} | {target_stats['mean_energy']:>12.6f} | {bridge_stats['mean_energy']:>12.6f}")
print("-" * 45)

from pymcd.mcd import Calculate_MCD

# Initialize the official metric calculator
# "dtw" mode automatically aligns the audio before scoring
mcd_toolbox = Calculate_MCD(MCD_mode="dtw")

# Calculate the true MCD
mcd_score = mcd_toolbox.calculate_mcd(REFERENCE_WAV, output_filename)

print(f"\n2️⃣ ACOUSTIC DISTORTION")
print("-" * 45)
print(f"📉 Official DTW-MCD Score:     {mcd_score:.2f} dB")

# --- C. ECAPA-TDNN IDENTITY CHECK ---
print(f"\n3️⃣ BIOMETRIC IDENTITY TRANSFER")
print("-" * 45)
try:
    verification = SpeakerRecognition.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb", 
        run_opts={"device": DEVICE_HOME}
    )
    wav_target = verification.load_audio(REFERENCE_WAV)
    wav_output = verification.load_audio(output_filename)
    
    emb1 = verification.encode_batch(wav_target)
    emb2 = verification.encode_batch(wav_output)
    
    similarity = torch.nn.functional.cosine_similarity(emb1.flatten(), emb2.flatten(), dim=0).item()
    print(f"🧬 ECAPA-TDNN Similarity:      {similarity:.4f}")
except Exception as e:
    print(f"⚠️ Identity Check Failed: {e}")

print("="*50)

In [ ]:
import torch
import torch.nn as nn
import torchaudio
import sentencepiece
import librosa
import numpy as np
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

# ==========================================
# 0. THE DYNAMIC GRAPH KILLER
# ==========================================
import moshi.utils.compile

print("🔍 Hunting for Kyutai's CUDA Graph compiler...")
patched_any = False
for name, obj in vars(moshi.utils.compile).items():
    if isinstance(obj, type) and hasattr(obj, '__call__'):
        def make_bypass(orig_call):
            def bypass_call(self, *args, **kwargs):
                if hasattr(self, 'func'): return self.func(*args, **kwargs)
                return orig_call(self, *args, **kwargs)
            return bypass_call
        obj.__call__ = make_bypass(obj.__call__)
        print(f"✅ Successfully neutralized CUDA Graphs in: moshi.utils.compile.{name}")
        patched_any = True

# ==========================================
# 1. SETUP, PATHS & V2.2 BRIDGE ARCHITECTURE
# ==========================================
DEVICE_HOME = "cuda:0" 
DEVICE_WORK = "cuda:1" 

# ⚠️ Make sure this points to your NEW V2.2 model once it finishes!
BRIDGE_PATH = "/kaggle/input/datasets/ahmedsadman099876/data888/acoustic_bridge_v2_final.pt" 
TARGET_PT_PATH = "/kaggle/input/datasets/ahmedsadman099876/data899/0012_001164.pt"
USER_PROMPT_WAV = "/kaggle/input/datasets/ahmedsadman099876/recording/Recording.wav"
REFERENCE_WAV = "/kaggle/input/datasets/ahmedsadman099876/data458/0012_001164.wav"


class AcousticDisentanglementBridge(nn.Module):
    def __init__(self, vocab_size=2048, embed_dim=1024, id_dim=192, max_len=4096):
        super().__init__()
        
        self.cb0_embedding = nn.Embedding(vocab_size, embed_dim)
        self.id_projection = nn.Linear(id_dim, embed_dim)
        self.positional_encoding = nn.Embedding(max_len, embed_dim)
        
        self.cnn_prenet = nn.Sequential(
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            nn.GELU()
        )
        
        self.temporal_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, batch_first=True, dropout=0.1), 
            num_layers=2
        )
        
        self.acoustic_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, embed_dim // 2),
                nn.GELU(),
                nn.LayerNorm(embed_dim // 2),
                nn.Linear(embed_dim // 2, vocab_size)
            ) for _ in range(7)
        ])

    def apply_ppmq_adain(self, cb0_emb, id_vec):
        id_raw = self.id_projection(id_vec).unsqueeze(1)
        id_min, id_max = id_raw.min(dim=-1, keepdim=True)[0], id_raw.max(dim=-1, keepdim=True)[0]
        scale = (id_max - id_min) / 15
        id_quant = torch.round((id_raw - id_min) / (scale + 1e-8)) * scale + id_min
        
        cb0_std, cb0_mean = cb0_emb.std(dim=-1, keepdim=True), cb0_emb.mean(dim=-1, keepdim=True)
        id_std, id_mean = id_quant.std(dim=-1, keepdim=True) + 1e-8, id_quant.mean(dim=-1, keepdim=True)
        
        return cb0_emb + ((cb0_std * ((id_quant - id_mean) / id_std) + cb0_mean) * 0.8)

    def forward(self, cb0_tokens, id_vec):
        if cb0_tokens.dim() == 3: cb0_tokens = cb0_tokens.view(cb0_tokens.shape[0], -1)
        seq_len = cb0_tokens.size(1)
        device = cb0_tokens.device
        
        positions = torch.arange(0, seq_len, device=device).unsqueeze(0)
        cb0_emb = self.cb0_embedding(cb0_tokens) + self.positional_encoding(positions)
        
        fused_emb = self.apply_ppmq_adain(cb0_emb, id_vec)
        # 🛡️ Safety wrapper kept intact for inference
        fused_emb = torch.nan_to_num(fused_emb, nan=0.0, posinf=1.0, neginf=-1.0)
        
        fused_emb = fused_emb.transpose(1, 2)
        fused_emb = self.cnn_prenet(fused_emb)
        fused_emb = fused_emb.transpose(1, 2)
        
        temporal_features = self.temporal_transformer(fused_emb)
        logits_list = [head(temporal_features) for head in self.acoustic_heads]
        return torch.stack(logits_list, dim=2)

# ==========================================
# 2. LOAD MODELS & CONSTRUCT SANDWICH
# ==========================================
from moshi.models import loaders, LMGen

repo_id = "kyutai/moshika-pytorch-bf16"
print("\nDownloading Mimi weights...")
mimi = loaders.get_mimi(hf_hub_download(repo_id, "tokenizer-e351c8d8-checkpoint125.safetensors"), device="cpu")
mimi = mimi.to(DEVICE_HOME)
mimi.set_num_codebooks(8)

print("Downloading Moshi weights (~15.4 GB)...")
moshi = loaders.get_moshi_lm(hf_hub_download(repo_id, "model.safetensors"), device="cpu")
text_tokenizer = sentencepiece.SentencePieceProcessor(hf_hub_download(repo_id, "tokenizer_spm_32k_3.model"))

print("\n🛠️ Sharding Model across GPUs...")
def shard_hook(module, args):
    try: target_dev = next(module.parameters()).device
    except StopIteration:
        try: target_dev = next(module.buffers()).device
        except StopIteration: return args
    if isinstance(args, tuple): return tuple(a.to(target_dev) if isinstance(a, torch.Tensor) else a for a in args)
    return args.to(target_dev) if isinstance(args, torch.Tensor) else args

mid = len(moshi.transformer.layers) // 2
for i in range(mid, len(moshi.transformer.layers)):
    moshi.transformer.layers[i].to(DEVICE_WORK)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)

moshi.emb.to(DEVICE_HOME)
moshi.text_emb.to(DEVICE_HOME)
for i in range(mid):
    moshi.transformer.layers[i].to(DEVICE_HOME)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)
if hasattr(moshi.transformer, "norm"):
    moshi.transformer.norm.to(DEVICE_HOME)
    moshi.transformer.norm.register_forward_pre_hook(shard_hook)
for name, module in moshi.named_children():
    if name not in ["emb", "text_emb", "transformer"]:
        module.to(DEVICE_HOME)
        module.register_forward_pre_hook(shard_hook)

for name, buf in moshi.named_buffers(recurse=True): buf.data = buf.data.to(DEVICE_HOME)
for attr in ['initial', 'delays', 'zero_token_id']:
    if hasattr(moshi, attr):
        t = getattr(moshi, attr)
        if isinstance(t, torch.Tensor): setattr(moshi, attr, t.to(DEVICE_HOME))

lm_gen = LMGen(moshi, temp=0.8, temp_text=0.7)

# ==========================================
# 3. PHASE 1: NATIVE GENERATION (SMART TRIMMER)
# ==========================================
try:
    wav, sr = torchaudio.load(USER_PROMPT_WAV)
    if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
    if sr != 24000: wav = torchaudio.functional.resample(wav, sr, 24000)
    wav = wav.unsqueeze(0).to(DEVICE_HOME)
except FileNotFoundError:
    wav = torch.zeros(1, 1, 24000 * 3).to(DEVICE_HOME)

print("\n⚡ Phase 1: Generating Base Tokens Natively...")
frame_size = mimi.frame_size
all_codes = []
with torch.no_grad(), mimi.streaming(batch_size=1):
    for offset in range(0, wav.shape[-1], frame_size):
        frame = wav[:, :, offset : offset + frame_size]
        if frame.shape[-1] < frame_size: frame = torch.nn.functional.pad(frame, (0, frame_size - frame.shape[-1]))
        all_codes.append(mimi.encode(frame))

collected_moshi_codes = []
generated_text_pieces = []

with torch.no_grad(), lm_gen.streaming(1), mimi.streaming(1):
    def process_step(tokens_out):
        text_token = tokens_out[0, 0].item()
        if text_token not in (0, 3): 
            generated_text_pieces.append(text_tokenizer.id_to_piece(text_token).replace(' ', ' '))
        collected_moshi_codes.append(tokens_out[:, 1:])

    for code in all_codes:
        tokens_out = lm_gen.step(code)
        if tokens_out is not None: process_step(tokens_out)

    print("Moshi is replying...")
    silence_code = torch.zeros_like(all_codes[0]).to(DEVICE_HOME)
    
    # --- SMART TAIL TRIMMER ---
    eos_detected = False
    tail_frames = 0
    MAX_TAIL = 10 
    
    for _ in range(120): 
        tokens_out = lm_gen.step(silence_code)
        if tokens_out is not None: 
            text_token = tokens_out[0, 0].item()
            
            if text_token in (0, 3):
                eos_detected = True
                
            process_step(tokens_out)
            
            if eos_detected:
                tail_frames += 1
                if tail_frames >= MAX_TAIL:
                    print(f"   [Sentence complete. Cutting audio after {MAX_TAIL} frames to prevent static.]")
                    break
    
final_moshi_codes = torch.cat(collected_moshi_codes, dim=-1) 

print("\n" + "="*40)
print("MOSHI'S TEXT RESPONSE:")
print("".join(generated_text_pieces).strip() or "(No text generated)")
print("="*40 + "\n")

# ==========================================
# 4. PHASE 2: BRIDGING & DSP POST-PROCESSING
# ==========================================
print("🌉 Phase 2: Applying Bridge and DSP Filters...")

bridge = AcousticDisentanglementBridge().to(DEVICE_HOME)

# Load the weights!
bridge.load_state_dict(torch.load(BRIDGE_PATH, map_location=DEVICE_HOME, weights_only=True))
bridge.eval()

sample_data = torch.load(TARGET_PT_PATH, weights_only=True)
id_vec = sample_data["identity_vector"].view(1, -1).to(DEVICE_HOME, dtype=torch.float32)

with torch.no_grad():
    cb0_sequence = final_moshi_codes[:, 0, :] 
    new_logits = bridge(cb0_sequence, id_vec) 
    
    new_cb1_7 = torch.argmax(new_logits, dim=-1).transpose(1, 2) 
    
    hybrid_codes = final_moshi_codes.clone()
    hybrid_codes[:, 1:8, :] = new_cb1_7
    
    bridged_waveform = mimi.decode(hybrid_codes).to(torch.float32).cpu()

# --- DSP TWEAK 1: RAISED LOWPASS FILTER ---
# Bumped to 7500Hz. Keeps the 'S' and 'F' sounds crisp, blocks the 10kHz+ static.
clean_waveform = torchaudio.functional.lowpass_biquad(bridged_waveform, sample_rate=24000, cutoff_freq=7500.0)

# --- DSP TWEAK 2: THE NATURAL FADE-OUT ---
# Fades the last 0.3 seconds to absolute zero to prevent abrupt cutting clicks
fade_samples = int(24000 * 0.3) # 0.3 seconds of fade
if clean_waveform.shape[-1] > fade_samples:
    fade_curve = torch.linspace(1.0, 0.0, fade_samples)
    clean_waveform[0, 0, -fade_samples:] *= fade_curve

output_filename = "/kaggle/working/final_bridged_eval_DSP_Tuned.wav"
torchaudio.save(output_filename, clean_waveform.squeeze(0), 24000)

print(f"🎉 Success! Bridged audio saved to: {output_filename}")
display(Audio(output_filename, rate=24000))

# ==========================================
# 5. EVALUATION (DTW-ALIGNED MCD)
# ==========================================
print("📊 Phase 3: Evaluating Aligned MCD Score...")

# Load audio at 16kHz
y_ref, _ = librosa.load(REFERENCE_WAV, sr=16000)
y_gen, _ = librosa.load(output_filename, sr=16000)

# Extract MFCCs
mfcc_ref = librosa.feature.mfcc(y=y_ref, sr=16000, n_mfcc=14)[1:, :]
mfcc_gen = librosa.feature.mfcc(y=y_gen, sr=16000, n_mfcc=14)[1:, :]

# Apply DTW
D, wp = librosa.sequence.dtw(X=mfcc_ref, Y=mfcc_gen, metric='euclidean')

mfcc_ref_aligned = mfcc_ref[:, wp[:, 0]]
mfcc_gen_aligned = mfcc_gen[:, wp[:, 1]]

diff = mfcc_ref_aligned - mfcc_gen_aligned
mcd = (10.0 / np.log(10)) * np.mean(np.sqrt(2 * np.sum(diff**2, axis=0)))

print(f"🎉 Success! Bridged audio saved to: {output_filename}")
print(f"📈 REALITY-CHECKED MCD SCORE: {mcd:.2f} dB")

# Create a playable audio widget in the Kaggle notebook
display(Audio(output_filename, rate=24000))

In [ ]:
import torch
import torch.nn as nn
import torchaudio
import librosa
import numpy as np
import os
from moshi.models import loaders
from huggingface_hub import hf_hub_download

# ==========================================
# 1. SETUP & DUAL-GPU CONFIG
# ==========================================
# The "Compute Sandwich" setup:
DEVICE_HOME = "cuda:0" # Entry/Exit logic, Context, Heads, and Bridge
DEVICE_WORK = "cuda:1" # Heavy lifting (Middle Transformer Layers)

BRIDGE_PATH = "/kaggle/input/datasets/ahmedsadman099876/data589/acoustic_bridge_final_bf16_aligned.pt"
TARGET_PT_PATH = "/kaggle/input/datasets/ahmedsadman099876/data899/0012_001164.pt"
USER_PROMPT_WAV = "/kaggle/input/datasets/ahmedsadman099876/recording/Recording.wav"
REFERENCE_WAV = "/kaggle/input/datasets/ahmedsadman099876/data458/0012_001164.wav"

class AcousticDisentanglementBridge(nn.Module):
    def __init__(self, vocab_size=2048, embed_dim=1024, id_dim=192):
        super().__init__()
        self.cb0_embedding = nn.Embedding(vocab_size, embed_dim)
        self.id_projection = nn.Linear(id_dim, embed_dim)
        self.temporal_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, batch_first=True), 
            num_layers=2
        )
        self.acoustic_heads = nn.ModuleList([nn.Linear(embed_dim, vocab_size) for _ in range(7)])

    def apply_ppmq_adain(self, cb0_emb, id_vec):
        id_raw = self.id_projection(id_vec).unsqueeze(1)
        id_min, id_max = id_raw.min(dim=-1, keepdim=True)[0], id_raw.max(dim=-1, keepdim=True)[0]
        scale = (id_max - id_min) / 15
        id_quant = torch.round((id_raw - id_min) / (scale + 1e-8)) * scale + id_min
        cb0_std, cb0_mean = cb0_emb.std(dim=-1, keepdim=True), cb0_emb.mean(dim=-1, keepdim=True)
        id_std, id_mean = id_quant.std(dim=-1, keepdim=True) + 1e-8, id_quant.mean(dim=-1, keepdim=True)
        return cb0_emb + ((cb0_std * ((id_quant - id_mean) / id_std) + cb0_mean) * 0.8)

    def forward(self, cb0_tokens, id_vec):
        if cb0_tokens.dim() == 3: cb0_tokens = cb0_tokens.view(cb0_tokens.shape[0], -1)
        cb0_emb = self.cb0_embedding(cb0_tokens)
        fused_emb = self.apply_ppmq_adain(cb0_emb, id_vec)
        
        # 🛡️ SAFETY CATCH: Prevent AdaIN math from blowing up into NaNs
        fused_emb = torch.nan_to_num(fused_emb, nan=0.0, posinf=1.0, neginf=-1.0)
        
        temporal_features = self.temporal_transformer(fused_emb)
        logits_list = [head(temporal_features) for head in self.acoustic_heads]
        return torch.stack(logits_list, dim=2)

# ==========================================
# 2. THE "COMPUTE SANDWICH" SHARDING
# ==========================================
print("📦 Loading Native BF16 Weights...")
mimi_path = hf_hub_download("kyutai/moshiko-pytorch-bf16", "tokenizer-e351c8d8-checkpoint125.safetensors")
moshi_path = hf_hub_download("kyutai/moshiko-pytorch-bf16", "model.safetensors")

# Auxiliary models securely on HOME GPU
mimi = loaders.get_mimi(mimi_path, device=DEVICE_HOME)
bridge = AcousticDisentanglementBridge().to(DEVICE_HOME)
bridge.load_state_dict(torch.load(BRIDGE_PATH, map_location=DEVICE_HOME, weights_only=True))
bridge.eval()

# Load LM to CPU first to dodge VRAM spikes
lm = loaders.get_moshi_lm(moshi_path, device="cpu")

def shard_hook(module, args):
    """Pulls data to the correct GPU dynamically."""
    try: target_dev = next(module.parameters()).device
    except StopIteration:
        try: target_dev = next(module.buffers()).device
        except StopIteration: return args
    if isinstance(args, tuple):
        return tuple(a.to(target_dev) if isinstance(a, torch.Tensor) else a for a in args)
    return args.to(target_dev) if isinstance(args, torch.Tensor) else args

print("🛠️ Constructing the Compute Sandwich...")

num_layers = len(lm.transformer.layers)
mid = num_layers // 2

# 1. MIDDLE: Offload heavy layers to WORKER GPU (cuda:1)
for i in range(mid, num_layers):
    lm.transformer.layers[i].to(DEVICE_WORK)
    lm.transformer.layers[i].register_forward_pre_hook(shard_hook)

# 2. BREAD: Move EVERYTHING ELSE to HOME GPU (cuda:0)
lm.emb.to(DEVICE_HOME)
lm.text_emb.to(DEVICE_HOME)

for i in range(mid):
    lm.transformer.layers[i].to(DEVICE_HOME)
    lm.transformer.layers[i].register_forward_pre_hook(shard_hook)

if hasattr(lm.transformer, "norm"):
    lm.transformer.norm.to(DEVICE_HOME)
    lm.transformer.norm.register_forward_pre_hook(shard_hook)

for name, module in lm.named_children():
    if name not in ["emb", "text_emb", "transformer"]:
        module.to(DEVICE_HOME)
        module.register_forward_pre_hook(shard_hook)

# 3. CRITICAL: Force all hidden buffers to HOME GPU to fix torch.cat mismatch
for name, buf in lm.named_buffers(recurse=True):
    buf.data = buf.data.to(DEVICE_HOME)
    
for attr in ['initial', 'delays', 'zero_token_id']:
    if hasattr(lm, attr):
        t = getattr(lm, attr)
        if isinstance(t, torch.Tensor):
            setattr(lm, attr, t.to(DEVICE_HOME))

# ==========================================
# 3. DUAL-GPU GENERATION LOOP (FIXED ALIGNMENT)
# ==========================================
print("🎤 Encoding Prompt...")
user_wav, sr = torchaudio.load(USER_PROMPT_WAV)
if sr != 24000: user_wav = torchaudio.functional.resample(user_wav, sr, 24000)
user_codes = mimi.encode(user_wav.unsqueeze(0).to(DEVICE_HOME))

T_init = user_codes.shape[-1]
full_context = torch.zeros((1, 17, T_init), dtype=torch.long, device=DEVICE_HOME)
full_context[:, 1:9, :] = user_codes

generated_audio_list = []

print("🚀 Generating Native Tokens...")
with torch.no_grad():
    for t in range(160):
        output = lm.forward(full_context)
        
        # 1. NATIVE SEPARATION: Text and Audio are separate in Native Moshi!
        audio_logits = output.logits
        
        # Safely pull text logits if available, otherwise fallback
        if hasattr(output, 'text_logits') and output.text_logits is not None:
            text_logits = output.text_logits
            if text_logits.dim() == 3:
                next_text = torch.argmax(text_logits[:, -1, :], dim=-1).unsqueeze(0)
            else:
                next_text = torch.argmax(text_logits[:, 0, -1, :], dim=-1)
        else:
            # Fallback if the model is unexpectedly wrapping text in the first channel
            next_text = torch.argmax(audio_logits[:, 0:1, -1, :], dim=-1)
            audio_logits = audio_logits[:, 1:] 
            
        # 2. Sample ALL Audio Codebooks
        next_moshi_audio = torch.argmax(audio_logits[:, :, -1, :], dim=-1)
        
        # Ensure it has exactly 8 channels for Mimi decoding
        num_audio = next_moshi_audio.shape[1]
        if num_audio < 8:
            pad = torch.zeros((1, 8 - num_audio), dtype=torch.long, device=DEVICE_HOME)
            moshi_8 = torch.cat([next_moshi_audio, pad], dim=1)
        else:
            moshi_8 = next_moshi_audio[:, :8]
            
        generated_audio_list.append(moshi_8.unsqueeze(-1))
        
        # 3. Build the next 17-channel frame perfectly
        new_frame = torch.zeros((1, 17, 1), dtype=torch.long, device=DEVICE_HOME)
        new_frame[:, 0, 0] = next_text
        new_frame[:, 9:9+num_audio, 0] = next_moshi_audio
        
        full_context = torch.cat([full_context, new_frame], dim=-1)
        
        if (t + 1) % 40 == 0: 
            print(f"✅ Frame {t+1}/160...")
            torch.cuda.empty_cache()

# ==========================================
# 4. DIAGNOSTIC BRIDGE & DECODING
# ==========================================
print("🌉 Applying Bridge & Decoding...")
final_moshi_codes = torch.cat(generated_audio_list, dim=-1)
sample_data = torch.load(TARGET_PT_PATH, weights_only=True)
id_vec = sample_data["identity_vector"].unsqueeze(0).to(DEVICE_HOME, dtype=torch.float32)

with torch.no_grad():
    pure_waveform = mimi.decode(final_moshi_codes).to(torch.float32).cpu()
    torchaudio.save("sandwich_pure_diagnostic.wav", pure_waveform.squeeze(0), 24000)
    
    cb0 = final_moshi_codes[:, 0, :]
    new_logits = bridge(cb0, id_vec)
    new_cb1_7 = torch.argmax(new_logits, dim=-1).transpose(1, 2)
    
    # --- 🔍 THE SMOKING GUN PRINT ---
    print("\n--- TOKEN DIAGNOSTIC ---")
    print(f"Native CB1 (Speech): {final_moshi_codes[0, 1, :15].tolist()}")
    print(f"Bridge CB1 (Output): {new_cb1_7[0, 0, :15].tolist()}")
    print("------------------------\n")
    
    hybrid_codes = final_moshi_codes.clone()
    hybrid_codes[:, 1:8, :] = new_cb1_7
    bridged_waveform = mimi.decode(hybrid_codes).to(torch.float32).cpu()

torchaudio.save("sandwich_final_bridged.wav", bridged_waveform.squeeze(0), 24000)

# ==========================================
# 5. EVALUATION
# ==========================================
y_ref, _ = librosa.load(REFERENCE_WAV, sr=16000)
y_gen, _ = librosa.load("sandwich_final_bridged.wav", sr=16000)

min_l = min(len(y_ref), len(y_gen))
mfcc_ref = librosa.feature.mfcc(y=y_ref[:min_l], sr=16000, n_mfcc=14)[1:, :]
mfcc_gen = librosa.feature.mfcc(y=y_gen[:min_l], sr=16000, n_mfcc=14)[1:, :]

diff = mfcc_ref - mfcc_gen
mcd = (10.0 / np.log(10)) * np.sqrt(2 * np.mean(np.sum(diff**2, axis=0)))

print(f"🎉 Success! Audio saved.")
print(f"📊 Final MCD Score: {mcd:.2f} dB")

In [ ]:
import torch
import torchaudio
import sentencepiece
from huggingface_hub import hf_hub_download

# ==========================================
# 0. THE DYNAMIC GRAPH KILLER
# ==========================================
import moshi.utils.compile

print("🔍 Hunting for Kyutai's CUDA Graph compiler...")
patched_any = False
# We scan every object inside their compile module
for name, obj in vars(moshi.utils.compile).items():
    # If we find a class with a __call__ method, it's their wrapper
    if isinstance(obj, type) and hasattr(obj, '__call__'):
        def make_bypass(orig_call):
            def bypass_call(self, *args, **kwargs):
                # If it has a .func, bypass the graph and just run the raw math!
                if hasattr(self, 'func'):
                    return self.func(*args, **kwargs)
                return orig_call(self, *args, **kwargs)
            return bypass_call
        
        obj.__call__ = make_bypass(obj.__call__)
        print(f"✅ Successfully neutralized CUDA Graphs in: moshi.utils.compile.{name}")
        patched_any = True

if not patched_any:
    print("⚠️ Warning: Could not locate compiler to patch.")

from moshi.models import loaders, LMGen
from IPython.display import Audio, display

# ==========================================
# 1. SETUP & DUAL-GPU CONFIG
# ==========================================
DEVICE_HOME = "cuda:0" # Entry/Exit logic, Memory, and Mimi
DEVICE_WORK = "cuda:1" # Heavy lifting (Middle Transformer Layers)

print(f"\n🚀 Initializing Dual-GPU Compute Sandwich...")
repo_id = "kyutai/moshiko-pytorch-bf16"
audio_path = "/kaggle/input/datasets/ahmedsadman099876/recording/Recording.wav"

# ==========================================
# 2. LOAD MODELS (CPU FIRST)
# ==========================================
print("\nDownloading Mimi weights...")
mimi = loaders.get_mimi(hf_hub_download(repo_id, "tokenizer-e351c8d8-checkpoint125.safetensors"), device="cpu")
mimi = mimi.to(DEVICE_HOME)
mimi.set_num_codebooks(8)

print("Downloading Moshi weights (~15.4 GB)...")
moshi = loaders.get_moshi_lm(hf_hub_download(repo_id, "model.safetensors"), device="cpu")

print("Downloading Text Tokenizer...")
text_tokenizer = sentencepiece.SentencePieceProcessor(hf_hub_download(repo_id, "tokenizer_spm_32k_3.model"))

# ==========================================
# 3. CONSTRUCT THE COMPUTE SANDWICH
# ==========================================
print("\n🛠️ Sharding Model across GPUs...")

def shard_hook(module, args):
    try: target_dev = next(module.parameters()).device
    except StopIteration:
        try: target_dev = next(module.buffers()).device
        except StopIteration: return args
    if isinstance(args, tuple):
        return tuple(a.to(target_dev) if isinstance(a, torch.Tensor) else a for a in args)
    return args.to(target_dev) if isinstance(args, torch.Tensor) else args

num_layers = len(moshi.transformer.layers)
mid = num_layers // 2

for i in range(mid, num_layers):
    moshi.transformer.layers[i].to(DEVICE_WORK)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)

moshi.emb.to(DEVICE_HOME)
moshi.text_emb.to(DEVICE_HOME)

for i in range(mid):
    moshi.transformer.layers[i].to(DEVICE_HOME)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)

if hasattr(moshi.transformer, "norm"):
    moshi.transformer.norm.to(DEVICE_HOME)
    moshi.transformer.norm.register_forward_pre_hook(shard_hook)

for name, module in moshi.named_children():
    if name not in ["emb", "text_emb", "transformer"]:
        module.to(DEVICE_HOME)
        module.register_forward_pre_hook(shard_hook)

for name, buf in moshi.named_buffers(recurse=True):
    buf.data = buf.data.to(DEVICE_HOME)
for attr in ['initial', 'delays', 'zero_token_id']:
    if hasattr(moshi, attr):
        t = getattr(moshi, attr)
        if isinstance(t, torch.Tensor):
            setattr(moshi, attr, t.to(DEVICE_HOME))

# Initialize LMGen with our patched environment
lm_gen = LMGen(moshi, temp=0.8, temp_text=0.7)

# ==========================================
# 4. PREPARE INPUT AUDIO
# ==========================================
try:
    print(f"\nProcessing audio: {audio_path}")
    wav, sr = torchaudio.load(audio_path)
    if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
    if sr != 24000: wav = torchaudio.functional.resample(wav, sr, 24000)
    wav = wav.unsqueeze(0).to(DEVICE_HOME)
except FileNotFoundError:
    print(f"⚠️ Error: '{audio_path}' not found. Using silence.")
    wav = torch.zeros(1, 1, 24000 * 3).to(DEVICE_HOME)

# ==========================================
# 5. FAST DUAL-GPU INFERENCE LOOP
# ==========================================
print("\n⚡ Listening and generating response natively... (GPU Accelerated)")
frame_size = mimi.frame_size
all_codes = []

with torch.no_grad(), mimi.streaming(batch_size=1):
    for offset in range(0, wav.shape[-1], frame_size):
        frame = wav[:, :, offset : offset + frame_size]
        if frame.shape[-1] < frame_size:
            frame = torch.nn.functional.pad(frame, (0, frame_size - frame.shape[-1]))
        codes = mimi.encode(frame)
        all_codes.append(codes) 

out_wav_chunks = []
generated_text_pieces = []

with torch.no_grad(), lm_gen.streaming(1), mimi.streaming(1):
    
    def process_text_token(tokens_out):
        text_token = tokens_out[0, 0].item()
        if text_token not in (0, 3): 
            piece = text_tokenizer.id_to_piece(text_token)
            generated_text_pieces.append(piece.replace(' ', ' '))

    for code in all_codes:
        tokens_out = lm_gen.step(code)
        if tokens_out is not None:
            process_text_token(tokens_out)
            wav_chunk = mimi.decode(tokens_out[:, 1:])
            out_wav_chunks.append(wav_chunk.cpu())

    silence_code = torch.zeros_like(all_codes[0]).to(DEVICE_HOME)
    for _ in range(60):  
        tokens_out = lm_gen.step(silence_code)
        if tokens_out is not None:
            process_text_token(tokens_out)
            wav_chunk = mimi.decode(tokens_out[:, 1:])
            out_wav_chunks.append(wav_chunk.cpu())

# ==========================================
# 6. OUTPUT & PLAYBACK
# ==========================================
if out_wav_chunks:
    output_wav = torch.cat(out_wav_chunks, dim=-1)
else:
    output_wav = torch.zeros(1, 1, 24000) 

full_text = "".join(generated_text_pieces).strip()
print("\n" + "="*40)
print("MOSHI'S TEXT RESPONSE:")
print(full_text if full_text else "(No text generated)")
print("="*40 + "\n")

output_filename = "/kaggle/working/dual_gpu_lmgen_reply.wav"
torchaudio.save(output_filename, output_wav.squeeze(0), 24000)
print(f"Saved reply audio to: {output_filename}")

display(Audio(output_wav.squeeze().numpy(), rate=24000))

In [ ]:
import torch
import torch.nn as nn
import torchaudio
import librosa
import numpy as np
from transformers import MoshiForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# ==========================================
# 1. SETUP & PATHS
# ==========================================
BRIDGE_PATH = "/kaggle/input/datasets/ahmedsadman099876/data589/acoustic_bridge_final_bf16_aligned.pt"
TARGET_PT_PATH = "/kaggle/input/datasets/ahmedsadman099876/data899/0012_001164.pt"
USER_PROMPT_WAV = "/kaggle/input/datasets/ahmedsadman099876/recording/Recording.wav"
REFERENCE_WAV = "/kaggle/input/datasets/ahmedsadman099876/data458/0012_001164.wav"

class AcousticDisentanglementBridge(nn.Module):
    def __init__(self, vocab_size=2048, embed_dim=1024, id_dim=192):
        super().__init__()
        self.vocab_size = vocab_size
        self.cb0_embedding = nn.Embedding(vocab_size, embed_dim)
        self.id_projection = nn.Linear(id_dim, embed_dim)
        self.temporal_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, batch_first=True), 
            num_layers=2
        )
        self.acoustic_heads = nn.ModuleList([nn.Linear(embed_dim, vocab_size) for _ in range(7)])

    def apply_ppmq_adain(self, cb0_emb, id_vec):
        id_raw = self.id_projection(id_vec).unsqueeze(1)
        id_min, id_max = id_raw.min(dim=-1, keepdim=True)[0], id_raw.max(dim=-1, keepdim=True)[0]
        scale = (id_max - id_min) / 15
        id_quant = torch.round((id_raw - id_min) / (scale + 1e-8)) * scale + id_min
        cb0_std, cb0_mean = cb0_emb.std(dim=-1, keepdim=True), cb0_emb.mean(dim=-1, keepdim=True)
        id_std, id_mean = id_quant.std(dim=-1, keepdim=True) + 1e-8, id_quant.mean(dim=-1, keepdim=True)
        
        # RESEARCH FIX: Must match the 0.8 alpha used in training
        return cb0_emb + ((cb0_std * ((id_quant - id_mean) / id_std) + cb0_mean) * 0.8)

    def forward(self, cb0_tokens, id_vec):
        if cb0_tokens.dim() == 3: cb0_tokens = cb0_tokens.view(cb0_tokens.shape[0], -1)
        cb0_emb = self.cb0_embedding(cb0_tokens)
        fused_emb = self.apply_ppmq_adain(cb0_emb, id_vec)
        temporal_features = self.temporal_transformer(fused_emb)
        logits_list = [head(temporal_features) for head in self.acoustic_heads]
        return torch.stack(logits_list, dim=2)

print("Loading 8-bit Base Model & Aligned Bridge...")
bnb_config_8bit = BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0)
processor = AutoProcessor.from_pretrained("kmhf/hf-moshiko")
model = MoshiForConditionalGeneration.from_pretrained(
    "kmhf/hf-moshiko", 
    quantization_config=bnb_config_8bit, 
    device_map={"": 0}, 
    torch_dtype=torch.float16
)

bridge = AcousticDisentanglementBridge().to("cuda:0")
bridge.load_state_dict(torch.load(BRIDGE_PATH, map_location="cuda:0", weights_only=True))
bridge.eval()

# ==========================================
# 2. GENERATION
# ==========================================
user_audio, sr = torchaudio.load(USER_PROMPT_WAV)
if sr != 24000: user_audio = torchaudio.functional.resample(user_audio, sr, 24000)
user_audio = user_audio.to("cuda:0", dtype=torch.float16).unsqueeze(0)

with torch.no_grad():
    encoded_input = model.audio_encoder.encode(user_audio)
    user_codes = encoded_input.audio_codes[:, :8, :] 
    unconditionals = model.get_unconditional_inputs()
    u_moshi, u_user, u_text = unconditionals["moshi_audio_codes"].to("cuda:0"), unconditionals["user_audio_codes"].to("cuda:0"), unconditionals["input_ids"].to("cuda:0")

    full_user_codes = torch.cat([u_user, user_codes], dim=2)
    full_moshi_codes = torch.cat([u_moshi, u_moshi.repeat(1, 1, user_codes.shape[-1])], dim=2)
    full_text_ids = torch.cat([u_text, u_text.repeat(1, user_codes.shape[-1])], dim=1)

    prompt_length = full_moshi_codes.shape[-1]
    
    outputs = model.generate(
        input_ids=full_text_ids,
        user_audio_codes=full_user_codes,
        moshi_audio_codes=full_moshi_codes,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.8,
        concat_unconditional_inputs=False,
        return_audio_codes=True  
    ) 
    
    generated_audio_codes = outputs.audio_codes[:, :, prompt_length:] 
    print(f"Isolated {generated_audio_codes.shape[-1]} frames.")

# ==========================================
# 3. INTERCEPTION & DECODING
# ==========================================
sample_data = torch.load(TARGET_PT_PATH, weights_only=True)
id_vec = sample_data["identity_vector"].unsqueeze(0).to("cuda:0", dtype=torch.float32)

with torch.no_grad():
    cb0 = generated_audio_codes[:, 0, :]
    new_acoustic_logits = bridge(cb0, id_vec) 
    
    # Deterministic Greedy Decoding
    new_cb1_7_tokens = torch.argmax(new_acoustic_logits, dim=-1).transpose(1, 2)
    
    # 16-Codebook Reconstruction: Replace 1-7, keep original 8-15
    hybrid_codes = generated_audio_codes.clone()
    hybrid_codes[:, 1:8, :] = new_cb1_7_tokens
    
    # FIXED DECODING
    waveform = model.audio_encoder.decode(hybrid_codes).audio_values.cpu().to(torch.float32)
    # Force it into a 2D tensor of shape [1, Time] for torchaudio
    if waveform.dim() > 2:
        waveform = waveform.squeeze() 
    if waveform.dim() == 1:
        waveform = waveform.unsqueeze(0)

output_path = "generative_bot_reply_raw.wav"
torchaudio.save(output_path, waveform, 24000)
print(f"Saved: {output_path}")

# ==========================================
# 4. EVALUATION
# ==========================================
y_ref, _ = librosa.load(REFERENCE_WAV, sr=16000)
y_gen, _ = librosa.load(output_path, sr=16000)
min_len = min(len(y_ref), len(y_gen))
mfcc_ref = librosa.feature.mfcc(y=y_ref[:min_len], sr=16000, n_mfcc=14)[1:, :]
mfcc_gen = librosa.feature.mfcc(y=y_gen[:min_len], sr=16000, n_mfcc=14)[1:, :]
mcd = (10.0 / np.log(10)) * np.sqrt(2 * np.sum((np.mean(mfcc_ref, axis=1) - np.mean(mfcc_gen, axis=1))**2))
print(f"MCD: {mcd:.2f} dB")

In [ ]:
import torch
import torch.nn as nn
import torchaudio
import librosa
import numpy as np
from transformers import MoshiForConditionalGeneration, BitsAndBytesConfig

# ==========================================
# 1. SETUP & PATHS
# ==========================================
BRIDGE_PATH = "/kaggle/input/datasets/ahmedsadman099876/data8999/acoustic_bridge_final.pt"
# The voice you want to copy (The Identity)
TARGET_PT_PATH = "/kaggle/input/datasets/ahmedsadman099876/data899/0012_001164.pt" 
# The audio you want to transform (The Source Semantics/Prosody)
SOURCE_WAV_PATH = "/kaggle/input/datasets/ahmedsadman099876/recording/Recording.wav" 
# The reference for MCD evaluation (Usually the Target Speaker's real audio)
REFERENCE_WAV = "/kaggle/input/datasets/ahmedsadman099876/data458/0012_001164.wav"

class AcousticDisentanglementBridge(nn.Module):
    def __init__(self, vocab_size=2048, embed_dim=1024, id_dim=192):
        super().__init__()
        self.vocab_size = vocab_size
        self.cb0_embedding = nn.Embedding(vocab_size, embed_dim)
        self.id_projection = nn.Linear(id_dim, embed_dim)
        self.temporal_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, batch_first=True), 
            num_layers=2
        )
        self.acoustic_heads = nn.ModuleList([
            nn.Linear(embed_dim, vocab_size) for _ in range(7)
        ])

    def apply_ppmq_adain(self, cb0_emb, id_vec):
        id_raw = self.id_projection(id_vec).unsqueeze(1)
        id_min, id_max = id_raw.min(dim=-1, keepdim=True)[0], id_raw.max(dim=-1, keepdim=True)[0]
        scale = (id_max - id_min) / 15
        id_quant = torch.round((id_raw - id_min) / (scale + 1e-8)) * scale + id_min
        cb0_std, cb0_mean = cb0_emb.std(dim=-1, keepdim=True), cb0_emb.mean(dim=-1, keepdim=True)
        id_std, id_mean = id_quant.std(dim=-1, keepdim=True) + 1e-8, id_quant.mean(dim=-1, keepdim=True)
        return cb0_emb + ((cb0_std * ((id_quant - id_mean) / id_std) + cb0_mean) * 0.1)

    def forward(self, cb0_tokens, id_vec):
        if cb0_tokens.dim() == 3:
            cb0_tokens = cb0_tokens.view(cb0_tokens.shape[0], -1)
        cb0_emb = self.cb0_embedding(cb0_tokens)
        fused_emb = self.apply_ppmq_adain(cb0_emb, id_vec)
        temporal_features = self.temporal_transformer(fused_emb)
        logits_list = [head(temporal_features) for head in self.acoustic_heads]
        return torch.stack(logits_list, dim=2)

print("Loading Base Architecture & Bridge...")
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
# Notice we don't even need the AutoProcessor anymore because we aren't using text!
model = MoshiForConditionalGeneration.from_pretrained("kmhf/hf-moshika", quantization_config=bnb_config, device_map={"": 0}, torch_dtype=torch.float16)

bridge = AcousticDisentanglementBridge().to("cuda:0")
bridge.load_state_dict(torch.load(BRIDGE_PATH, map_location="cuda:0", weights_only=True))
bridge.eval()

# ==========================================
# 2. PHASE 1: ENCODING SOURCE AUDIO (Extracting Prosody)
# ==========================================
print("Phase 1: Extracting Source Semantics & Prosody (Codebook 0)...")
source_audio, sr = torchaudio.load(SOURCE_WAV_PATH)
if sr != 24000:
    source_audio = torchaudio.functional.resample(source_audio, sr, 24000)

# Ensure it's [Batch, Channels, Time]
if source_audio.dim() == 1:
    source_audio = source_audio.unsqueeze(0).unsqueeze(0)
elif source_audio.dim() == 2:
    source_audio = source_audio.unsqueeze(0)

source_audio = source_audio.to("cuda:0", dtype=torch.float16)

with torch.no_grad():
    # Pass the raw wav into Mimi to get the 8 latent codebooks
    encoded_outputs = model.audio_encoder.encode(source_audio)
    original_audio_codes = encoded_outputs.audio_codes[:, :8, :] # [1, 8, Seq]

# ==========================================
# 3. PHASE 2: DISENTANGLED LATENT INJECTION
# ==========================================
print("Phase 2: Overwriting Acoustic Identity with Temperature Sampling...")
sample_data = torch.load(TARGET_PT_PATH, weights_only=True)
id_vec = sample_data["identity_vector"].unsqueeze(0).to("cuda:0", dtype=torch.float32)

cb0_tokens = original_audio_codes[:, 0, :] # [1, Seq]

with torch.no_grad():
    new_acoustic_logits = bridge(cb0_tokens, id_vec) # [1, Seq, 7, 2048]
    
    # --- THE FIX: TEMPERATURE SAMPLING ---
    temperature = 0.5  # Lower = more robotic/stable, Higher = more chaotic/breath-like
    
    # Scale the logits by the temperature
    logits_scaled = new_acoustic_logits / temperature
    probs = torch.nn.functional.softmax(logits_scaled, dim=-1)
    
    # Reshape for multinomial sampling
    B, Seq, NumCB, Vocab = probs.shape
    probs_flat = probs.view(-1, Vocab)
    
    # Sample from the probability distribution instead of forcing Argmax
    sampled_flat = torch.multinomial(probs_flat, num_samples=1)
    
    # Reshape back to the required Mimi format [1, 7, Seq]
    new_cb1_7_tokens = sampled_flat.view(B, Seq, NumCB).transpose(1, 2)
    # -------------------------------------
    
    hybrid_audio_codes = torch.cat([cb0_tokens.unsqueeze(1), new_cb1_7_tokens], dim=1) # [1, 8, Seq]
# ==========================================
# 4. PHASE 3: MIMI DECODING (RAW, UNPRUNED)
# ==========================================
print("Phase 3: Synthesizing Final Wav-to-Wav Audio (Raw Output)...")
with torch.no_grad():
    waveform = model.audio_encoder.decode(hybrid_audio_codes).audio_values

waveform = waveform[0].cpu().to(torch.float32)
if waveform.dim() == 1: waveform = waveform.unsqueeze(0)

# NO NORMALIZATION. Saving the raw mathematical output directly.
output_path = "wav_to_wav_disentangled_raw.wav"
torchaudio.save(output_path, waveform, 24000)
print(f"SUCCESS: Raw, untampered audio saved to {output_path}.")

# ==========================================
# 5. MCD EVALUATION (RAW FULL-LENGTH)
# ==========================================
def calculate_mcd_raw(ref_path, gen_path):
    print("Calculating Raw, Unpruned Timbre Distance...")
    y_ref, sr_ref = librosa.load(ref_path, sr=16000)
    y_gen, sr_gen = librosa.load(gen_path, sr=16000)
    
    # NO SILENCE TRIMMING. Evaluating exactly what the model spit out.
    if len(y_gen) == 0: return float('nan')
    
    # Ensure arrays are the same length for a 1-to-1 comparison
    min_len = min(len(y_ref), len(y_gen))
    y_ref = y_ref[:min_len]
    y_gen = y_gen[:min_len]
    
    mfcc_ref = librosa.feature.mfcc(y=y_ref, sr=sr_ref, n_mfcc=14)[1:, :]
    mfcc_gen = librosa.feature.mfcc(y=y_gen, sr=sr_gen, n_mfcc=14)[1:, :]
    
    diff = np.mean(mfcc_ref, axis=1) - np.mean(mfcc_gen, axis=1)
    return (10.0 / np.log(10)) * np.sqrt(2 * np.sum(diff ** 2))

mcd_score = calculate_mcd_raw(REFERENCE_WAV, output_path)
print(f"Raw Mel-Cepstral Distortion: {mcd_score:.2f} dB")